In [ ]:
import logging
from corriscope.fpga_array import FPGAArray
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9.5, 6)

# Allow all logging level messages to show up
log=logging.getLogger()
log.setLevel(logging.DEBUG)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

    
def _tqdm(iterable, **kwargs):
    return tqdm(iterable, **kwargs) if tqdm is not None else iterable


%matplotlib ipympl 

In [ ]:
import pickle as pkl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

C0 = 299_792_458.0  # m/s


# -------------------------
# Core helpers (Py3.8-safe)
# -------------------------

def coord_from_stream(stream):
    slot, chan = stream // 16, stream % 16
    return slot, chan

def parabolic_subbin(y, k):
    if k <= 0 or k >= len(y) - 1:
        return 0.0
    y0, y1, y2 = y[k - 1], y[k], y[k + 1]
    denom = (y0 - 2.0 * y1 + y2)
    if denom == 0:
        return 0.0
    return 0.5 * (y0 - y2) / denom


def _next_pow2(n):
    n = int(n)
    if n <= 1:
        return 1
    return 1 << ((n - 1).bit_length())


def _nonoverlap_segments(nsamp, n_segments):
    nsamp = int(nsamp)
    if n_segments is None or int(n_segments) <= 1:
        return nsamp, np.array([0], dtype=int)
    n_segments = int(n_segments)
    Nseg = nsamp // n_segments
    if Nseg < 1:
        raise ValueError("n_segments too large for nsamp.")
    starts = np.arange(n_segments, dtype=int) * Nseg
    return Nseg, starts
    

def _choose_nonoverlap_segments(nsamp, n_segments, Nseg=None):
    nsamp = int(nsamp)

    if Nseg is not None:
        Nseg0 = int(Nseg)
        if Nseg0 <= 0 or Nseg0 > nsamp:
            raise ValueError("Nseg must be in [1, nsamp].")
        nfit = nsamp // Nseg0
        if nfit < 1:
            raise ValueError("Nseg too large; no segments fit.")
        starts = (np.arange(nfit, dtype=int) * Nseg0)
        return Nseg0, starts

    if n_segments is None or int(n_segments) <= 1:
        return nsamp, np.array([0], dtype=int)

    n_segments = int(n_segments)
    Nseg0 = nsamp // n_segments
    if Nseg0 < 1:
        raise ValueError("n_segments too large for nsamp.")
    Nseg0 = max(256, (Nseg0 // 256) * 256) if Nseg0 >= 256 else Nseg0
    nfit = nsamp // Nseg0
    starts = (np.arange(nfit, dtype=int) * Nseg0)
    return Nseg0, starts


def _auto_fft_lengths(Nseg, segmented, Nfft=None, up_factor=16, refine=True):
    if Nfft is None:
        Nfft0 = _next_pow2(Nseg if not segmented else (2 * Nseg))
    else:
        Nfft0 = int(Nfft)
        if Nfft0 < Nseg:
            raise ValueError("Nfft must be >= Nseg.")

    up = int(up_factor) if refine else 1
    Ncorr = up * Nfft0
    return Nfft0, up, Ncorr


def _lag_window_indices(lags, lag_window):
    if lag_window is None:
        return slice(None), lags
    lo, hi = lag_window
    msk = (lags >= lo) & (lags <= hi)
    idx = np.nonzero(msk)[0]
    if idx.size == 0:
        raise ValueError("lag_window produced empty selection.")
    return idx, lags[idx]


def _compute_segment_ffts(d, sel_idx, frame_idx, starts, Nseg, Nfft, window):
    M = len(sel_idx)
    nseg = int(starts.size)
    print('nfft',Nfft)
    X = np.empty((M, nseg, Nfft // 2 + 1), dtype=np.complex64)
    for si, idx in enumerate(sel_idx):
        xfull = d[idx, frame_idx, :]
        for m, st in enumerate(starts):
            seg = xfull[st:st + Nseg]
            segw = (seg - seg.mean()) * window
            X[si, m, :] = np.fft.rfft(segw, n=Nfft)
    return X


def _pair_list(M):
    return [(i, j) for i in range(M) for j in range(i + 1, M)]


def _pair_tau_from_X(X, pairs, Ncorr, lags, lag_idx, fs, refine, up):
    P = len(pairs)
    if isinstance(lag_idx, slice):
        n_lag = len(lags)
    else:
        n_lag = len(lag_idx)

    C_disp = np.empty((P, n_lag), dtype=np.float32)
    peak_lag_samp = np.empty(P, dtype=np.float32)
    peak_tau_ns = np.empty(P, dtype=np.float32)
    peak_amp = np.empty(P, dtype=np.float32)

    print('ncorr',Ncorr)
    for p, (i, j) in enumerate(pairs):
        Gbar = np.mean(X[i, :, :] * np.conjugate(X[j, :, :]), axis=0)
        r = np.fft.ifft(Gbar, n=Ncorr)
        mag = np.abs(np.fft.fftshift(r))

        mag_w = mag[lag_idx]
        C_disp[p, :] = mag_w.astype(np.float32, copy=False)

        kw = int(np.argmax(mag_w))
        k_full = int(kw if isinstance(lag_idx, slice) else lag_idx[kw])

        if refine and (0 < k_full < len(mag) - 1):
            delta = parabolic_subbin(mag, k_full)
            lag_hat = lags[k_full] + delta / float(up)
        else:
            lag_hat = lags[k_full]

        peak_lag_samp[p] = lag_hat
        peak_tau_ns[p] = (lag_hat / fs) * 1e9
        peak_amp[p] = mag[k_full]

    return C_disp, peak_lag_samp, peak_tau_ns, peak_amp


def _lags_samples_to_ns(lags_samp, fs_hz):
    """Convert lag axis in samples to ns."""
    return (np.asarray(lags_samp, dtype=np.float64) / float(fs_hz)) * 1e9


# ---------------------------------------------
# 1) Lag spectra products (unique i<j products)
# ---------------------------------------------

def plot_lags_products_unique(
    d, streams, boards=None, chans=None,
    fs=3.2e9, frame_idx=0,

    n_segments=None,
    Nseg=None,

    Nfft=None,
    refine=True,
    up_factor=16,
    lag_window=(-40, 40),

    hist_bins=120,
    show_hist=True,
    show_heatmap=True,
    show_spectra=False,
    spectra_max=None,
    spectra_alpha=0.15,
    heatmap_logscale=False,
):
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamples)")
    n_streams, ncap, nsamp = d.shape
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")

    # select streams
    sel_idx = []
    labels = []
    for i, s in enumerate(streams):
        board, chan = coord_from_stream(s)
        if (boards is None or board in boards) and (chans is None or chan in chans):
            sel_idx.append(i)
            labels.append((board, chan))
    M = len(sel_idx)
    if M < 2:
        raise ValueError("Need at least 2 selected streams.")

    # segmentation (non-overlap)
    Nseg0, starts = _choose_nonoverlap_segments(nsamp, n_segments, Nseg=Nseg)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg0).astype(np.float32)

    # FFT lengths
    Nfft0, up, Ncorr = _auto_fft_lengths(Nseg0, segmented, Nfft=Nfft, up_factor=up_factor, refine=refine)
    lags = (np.arange(Ncorr) - Ncorr // 2) / float(up)  # samples

    # lag window indices
    lag_idx, lags_disp = _lag_window_indices(lags, lag_window)  # samples
    lags_disp_ns = _lags_samples_to_ns(lags_disp, fs)           # ns (for plotting)

    # compute FFTs
    X = _compute_segment_ffts(d, sel_idx, frame_idx, starts, Nseg0, Nfft0, window)

    # pairs + delays
    pairs = _pair_list(M)
    C_disp, peak_lag_samp, peak_tau_ns, peak_amp = _pair_tau_from_X(
        X, pairs, Ncorr, lags, lag_idx, fs, refine, up
    )

    # sort: primary peak lag, tie-break by -amp
    order = np.lexsort((-peak_amp, peak_lag_samp))

    # histogram
    if show_hist:
        plt.figure(figsize=(8, 4))
        plt.hist(peak_tau_ns, bins=hist_bins)
        plt.xlabel("Peak delay [ns]")
        plt.ylabel("Count (products)")
        plt.title("Peak-delay histogram (unique pairs; P={})".format(len(pairs)))
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # spectra overplot (sorted+colored) — now x-axis in ns, no colorbar
    if show_spectra:
        idx_plot = order
        if spectra_max is not None and int(spectra_max) < len(idx_plot):
            rng = np.random.default_rng(0)
            idx_plot = rng.choice(idx_plot, size=int(spectra_max), replace=False)
            idx_plot = idx_plot[np.argsort(peak_lag_samp[idx_plot])]

        vmin, vmax = float(np.min(peak_lag_samp)), float(np.max(peak_lag_samp))
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.cm.coolwarm

        fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
        for p in idx_plot:
            ax.plot(
                lags_disp_ns,
                C_disp[p, :],
                color=cmap(norm(float(peak_lag_samp[p]))),
                alpha=spectra_alpha,
                linewidth=1.0,
            )
        ax.set_xlabel("Lag [ns]")
        ax.set_ylabel("|cross-correlation|")
        ax.set_title("Lag spectra (unique pairs; P={})".format(len(pairs)))
        ax.grid(True)
        plt.show()

    # heatmap (sorted rows) — x-axis in ns, colorbar label explicit
    if show_heatmap:
        C_show = C_disp[order, :]

        if heatmap_logscale:
            pos = C_show[C_show > 0]
            vmin = np.percentile(pos, 1) if pos.size else 1e-12
            vmax = np.percentile(C_show, 99)
            norm = LogNorm(vmin=max(vmin, 1e-12), vmax=max(vmax, max(vmin, 1e-12) * 1.01))
            cbar_label = "Correlation magnitude |r| (log)"
        else:
            norm = None
            cbar_label = "Correlation magnitude |r|"

        fig, ax = plt.subplots(figsize=(8, 10), constrained_layout=True)
        im = ax.imshow(
            C_show,
            aspect="auto",
            origin="lower",
            extent=[lags_disp_ns[0], lags_disp_ns[-1], 0, len(pairs)],
            interpolation="nearest",
            norm=norm,
        )
        ax.set_xlabel("Lag [ns]")
        ax.set_ylabel("Lag Product index (sorted by peak location)")
        ax.set_title("2D Histogram of the lag spectrum (unique pairs; P={})".format(len(pairs)))
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label(cbar_label)
        plt.show()

    return {
        "lags_disp_samples": lags_disp,
        "lags_disp_ns": lags_disp_ns,
        "corr_mag_disp": C_disp,          # unsorted product order
        "order": order,
        "peak_tau_ns": peak_tau_ns,
        "peak_lag_samples": peak_lag_samp,
        "peak_amp": peak_amp,
        "prod_ij": pairs,
        "labels": labels,
        "sel_idx": sel_idx,
        "segmentation": {
            "nsamp": int(nsamp),
            "Nseg": int(Nseg0),
            "starts": starts,
            "n_segments_effective": int(nseg_eff),
            "Nfft": int(Nfft0),
            "Ncorr": int(Ncorr),
            "refine": bool(refine),
            "up": int(up),
            "fs_hz": float(fs),
        },
    }


# -------------------------------------------------------
# 2) Closure (phase-closure-style) histogram from lag peaks
# -------------------------------------------------------
def closure_histogram_from_lag_peaks(
    d, streams,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,

    n_segments=None,
    Nseg=None,

    Nfft=None,
    refine=True,
    up_factor=16,
    lag_window=(-75, 75),

    hist_bins=200,
    show_debug=False,
):
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamples)")
    n_streams, ncap, nsamp = d.shape
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")

    # select streams
    sel_idx = []
    sel_labels = []
    for i, s in enumerate(streams):
        board, chan = coord_from_stream(s)
        if (boards is None or board in boards) and (chans is None or chan in chans):
            sel_idx.append(i)
            sel_labels.append((board, chan))
    M = len(sel_idx)
    if M < 2:
        raise ValueError("Need at least 2 selected streams.")

    # segmentation (non-overlap)
    Nseg0, starts = _choose_nonoverlap_segments(nsamp, n_segments, Nseg=Nseg)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg0).astype(np.float32)

    # FFT/correlation lengths
    Nfft0, up, Ncorr = _auto_fft_lengths(Nseg0, segmented, Nfft=Nfft, up_factor=up_factor, refine=refine)
    lags = (np.arange(Ncorr) - Ncorr // 2) / float(up)  # samples

    # lag window
    lag_idx, _lags_disp = _lag_window_indices(lags, lag_window)

    # FFTs for selected streams
    X = _compute_segment_ffts(d, sel_idx, frame_idx, starts, Nseg0, Nfft0, window)

    # compute tau for all pairs and store antisymmetrically (ps)
    tau_ij_ps = {}
    pairs = _pair_list(M)
    for (i, j) in pairs:
        Gbar = np.mean(X[i, :, :] * np.conjugate(X[j, :, :]), axis=0)
        r = np.fft.ifft(Gbar, n=Ncorr)
        mag = np.abs(np.fft.fftshift(r))

        mag_w = mag[lag_idx]
        kw = int(np.argmax(mag_w))
        k_full = int(kw if isinstance(lag_idx, slice) else lag_idx[kw])

        if refine and (0 < k_full < len(mag) - 1):
            delta = parabolic_subbin(mag, k_full)
            lag_hat = lags[k_full] + delta / float(up)
        else:
            lag_hat = lags[k_full]

        tau_hat_ps = (lag_hat / fs) * 1e12
        tau_ij_ps[(i, j)] = tau_hat_ps
        tau_ij_ps[(j, i)] = -tau_hat_ps

    # reference delays tau0[i] = tau(0,i) in ps
    tau0 = np.zeros(M, dtype=np.float64)
    for i in range(1, M):
        tau0[i] = tau_ij_ps[(0, i)]

    # residuals for i<j, i,j != 0 (ps)
    residuals = []
    for i in range(1, M):
        for j in range(i + 1, M):
            tau_meas = tau_ij_ps[(i, j)]
            tau_pred = tau0[j] - tau0[i]
            residuals.append(tau_meas - tau_pred)
    residuals = np.asarray(residuals, dtype=np.float64)

    expected = (M - 1) * (M - 2) // 2
    if show_debug:
        b0, s0 = sel_labels[0]
        print("Reference (selected index 0): board {}, slot/chan {}".format(b0, s0))
        print("Selected streams: {}".format(M))
        print("Segmentation: nsamp={}, Nseg={}, nseg_eff={}, Nfft={}, up={}".format(
            nsamp, Nseg0, nseg_eff, Nfft0, up
        ))
        print("Residual count: {} (expected {})".format(len(residuals), expected))
        if len(residuals) > 1:
            print("Residual mean {:.6g} ps, std {:.6g} ps".format(
                float(residuals.mean()), float(residuals.std(ddof=1))
            ))

    plt.figure(figsize=(8, 4))
    plt.hist(residuals, bins=hist_bins)
    plt.xlabel("Closure residual [ps]")
    plt.ylabel("Count")
    plt.title("Closure residuals (unique pairs; P={})".format(len(pairs)))
    plt.grid(True)
    plt.xlim(-5,5)
    plt.tight_layout()
    plt.show()

    return {
        "residuals_ps": residuals,
        "tau0_ps": tau0,
        "n_streams_selected": M,
        "sel_idx": sel_idx,
        "labels_selected": sel_labels,
        "refine": bool(refine),
        "up": int(up),
        "lag_window": lag_window,
        "segmentation": {
            "nsamp": int(nsamp),
            "Nseg": int(Nseg0),
            "starts": starts,
            "n_segments_effective": int(nseg_eff),
            "Nfft": int(Nfft0),
            "Ncorr": int(Ncorr),
            "fs_hz": float(fs),
        },
    }


# ---------------------------------------------------
# 3) Convenience wrapper to run both analyses together
# ---------------------------------------------------

def lag_spectra_and_closure_report(
    d, streams,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=None,
    Nseg=None,
    Nfft=None,
    refine=True,
    up_factor=16,
    lag_window_spectra=(-40, 40),
    lag_window_closure=(-200, 200),
    hist_bins_spectra=120,
    hist_bins_closure=200,
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.15,
    heatmap_logscale=False,
    show_debug=False,
):
    spectra_res = plot_lags_products_unique(
        d, streams, boards=boards, chans=chans,
        fs=fs, frame_idx=frame_idx,
        n_segments=n_segments, Nseg=Nseg,
        Nfft=Nfft, refine=refine, up_factor=up_factor,
        lag_window=lag_window_spectra,
        hist_bins=hist_bins_spectra,
        show_hist=True,
        show_heatmap=show_heatmap,
        show_spectra=show_spectra,
        spectra_max=spectra_max,
        spectra_alpha=spectra_alpha,
        heatmap_logscale=heatmap_logscale,
    )

    closure_res = closure_histogram_from_lag_peaks(
        d, streams, boards=boards, chans=chans,
        fs=fs, frame_idx=frame_idx,
        n_segments=n_segments, Nseg=Nseg,
        Nfft=Nfft, refine=refine, up_factor=up_factor,
        lag_window=lag_window_closure,
        hist_bins=hist_bins_closure,
        show_debug=show_debug,
    )

    return {"spectra": spectra_res, "closure": closure_res}

def estimate_tau_ref_ns(
    d, fs,
    ref_idx=0,
    frame_idx=0,
    n_segments=None,
    Nfft=None,
    lag_window=(-200, 200),
    refine=True,
    up_factor=16,
):
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamp)")
    n_streams, ncap, nsamp = d.shape
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")
    if ref_idx < 0 or ref_idx >= n_streams:
        raise ValueError("ref_idx out of range")

    Nseg, starts = _nonoverlap_segments(nsamp, n_segments)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg).astype(np.float32)

    if Nfft is None:
        Nfft0 = _next_pow2(Nseg if not segmented else (2 * Nseg))
    else:
        Nfft0 = int(Nfft)
        if Nfft0 < Nseg:
            raise ValueError("Nfft must be >= Nseg")

    up = int(up_factor) if refine else 1
    Ncorr = up * Nfft0
    lags = (np.arange(Ncorr) - Ncorr // 2) / float(up)

    if lag_window is None:
        lag_idx = slice(None)
    else:
        lo, hi = lag_window
        msk = (lags >= lo) & (lags <= hi)
        lag_idx = np.nonzero(msk)[0]
        if lag_idx.size == 0:
            raise ValueError("lag_window produced empty selection.")

    X = np.empty((n_streams, nseg_eff, Nfft0 // 2 + 1), dtype=np.complex64)
    for i in range(n_streams):
        xfull = d[i, frame_idx, :]
        for m, st in enumerate(starts):
            seg = xfull[st:st + Nseg]
            segw = (seg - seg.mean()) * window
            X[i, m, :] = np.fft.rfft(segw, n=Nfft0)

    tau_ns = np.zeros(n_streams, dtype=np.float64)
    tau_ns[ref_idx] = 0.0

    for i in range(n_streams):
        if i == ref_idx:
            continue
        Gbar = np.mean(X[ref_idx, :, :] * np.conjugate(X[i, :, :]), axis=0)
        r = np.fft.ifft(Gbar, n=Ncorr)
        mag = np.abs(np.fft.fftshift(r))

        mag_w = mag[lag_idx]
        kw = int(np.argmax(mag_w))
        k_full = int(kw if isinstance(lag_idx, slice) else lag_idx[kw])

        if refine and (0 < k_full < len(mag) - 1):
            delta = parabolic_subbin(mag, k_full)
            lag_hat = lags[k_full] + delta / float(up)
        else:
            lag_hat = lags[k_full]

        tau_ns[i] = (lag_hat / fs) * 1e9

    return tau_ns


def build_effective_geometry_from_tau(tau_ns, aperture_m=None, center=True):
    tau_s = tau_ns * 1e-9
    if center:
        tau_s = tau_s - np.mean(tau_s)
    x = C0 * tau_s
    if aperture_m is not None:
        span = float(np.max(x) - np.min(x))
        if span <= 0:
            raise ValueError("Degenerate geometry span.")
        x = x * (float(aperture_m) / span)
    return x


def wideband_delay_and_sum_beam_pattern(
    d, fs, x_m,
    frame_idx=0,
    n_segments=None,
    Nfft=None,
    theta_deg=None,
    n_nyqzone=1,
    fmin=100e6, fmax=1.6e9,
    show_progress=True,
    progress_desc="Beam scan",
):
    if theta_deg is None:
        theta_deg = np.linspace(-90, 90, 721)

    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamp)")
    n_streams, ncap, nsamp = d.shape
    if len(x_m) != n_streams:
        raise ValueError("x_m must have length n_streams.")
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")

    Nseg, starts = _nonoverlap_segments(nsamp, n_segments)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg).astype(np.float32)

    if Nfft is None:
        Nfft0 = _next_pow2(Nseg if not segmented else (2 * Nseg))
    else:
        Nfft0 = int(Nfft)
        if Nfft0 < Nseg:
            raise ValueError("Nfft must be >= Nseg")

    if n_nyqzone % 2 == 0:
        fnyq = fs * n_nyqzone / 2
    else:
        fnyq = fs * (n_nyqzone - 1) / 2

    fnyq_sign = ( ( n_nyqzone % 2 ) * 2 - 1 )
    
    freqs = fnyq + fnyq_sign * np.fft.rfftfreq(Nfft0, d=1.0 / fs)
    
    print(freqs)
    fmask = (freqs >= fmin) & (freqs <= fmax)
    freqs_use = freqs[fmask]
    if freqs_use.size < 2:
        raise ValueError("Band selection empty; check fmin/fmax and Nfft.")

    X = np.empty((n_streams, nseg_eff, freqs_use.size), dtype=np.complex64)
    for i in range(n_streams):
        xfull = d[i, frame_idx, :]
        for m, st in enumerate(starts):
            seg = xfull[st:st + Nseg]
            segw = (seg - seg.mean()) * window
            Xi = np.fft.rfft(segw, n=Nfft0)
            X[i, m, :] = Xi[fmask]

    thetas = np.deg2rad(np.asarray(theta_deg, dtype=float))
    P = np.zeros_like(thetas, dtype=np.float64)

    iterator = list(enumerate(thetas))
    if show_progress and tqdm is not None:
        iterator = tqdm(iterator, total=len(thetas), desc=progress_desc, unit="theta", leave=True)

    for ti, th in iterator:
        tau = -(x_m * np.sin(th)) / C0
        W = np.exp(fnyq_sign*1j * 2 * np.pi * freqs_use[None, :] * tau[:, None]).astype(np.complex64)

        acc = 0.0
        for m in range(nseg_eff):
            Yf = np.sum(W * X[:, m, :], axis=0)
            acc += np.sum(np.abs(Yf) ** 2)
        P[ti] = acc / float(nseg_eff)

    return np.asarray(theta_deg, dtype=float), P


def azel_and_ewns_heatmaps(
    d_x, streams_x,
    d_y=None, streams_y=None,
    fs=3.2e9,
    frame_idx_x=0,
    frame_idx_y=0,

    # If d_y is None: slice d_x into two legs using nx, ny
    nx=32, ny=32,

    n_segments=None,
    Nfft=None,

    lag_window=(-200, 200),
    refine=True,
    up_factor=16,

    aperture_x_m=100.0,
    aperture_y_m=100.0,
    sort_by_pos=True,

    theta_x_deg=np.linspace(-90, 90, 721),
    theta_y_deg=np.linspace(-90, 90, 721),

    n_nyqzone=1,    
    fmin=100e6, fmax=1.6e9,
    show_progress=True,
    heatmap_db=True,

    az_sph_deg=np.linspace(0, 360, 721),
    el_sph_deg=np.linspace(0, 90, 361),

    show_slices=True,
):
    """
    Produce BOTH heatmaps:
      (1) EW/NS map in scan-angle space (θx vs θy), where u=sin(θx), v=sin(θy)
      (2) Spherical AZ/EL map (az 0–360, el 0–90) via direction-cosine mapping

    Input modes
    ----------
    Mode A (single capture):
        azel_and_ewns_heatmaps(d, streams)
      Interprets d as containing BOTH legs by slicing:
        d_x = d[:nx], d_y = d[nx:nx+ny]

    Mode B (two independent legs):
        azel_and_ewns_heatmaps(d_x, streams_x, d_y, streams_y)
      Uses d_x for EW leg and d_y for NS leg, with independent frame indices.

    Also generates 1D slice plots through the 2D heatmaps at the peak.
    """

    # ----- normalize inputs into two legs -----
    if d_y is None:
        # slice single capture
        d = d_x
        streams = streams_x
        if d.ndim != 3:
            raise ValueError("Expected d shape (n_streams, ncap, nsamp)")
        if d.shape[0] < nx + ny:
            raise ValueError("Need at least nx+ny streams in single-capture mode.")
        d_x = d[:nx, :, :]
        d_y = d[nx:nx + ny, :, :]
        streams_x = streams[:nx]
        streams_y = streams[nx:nx + ny]
        frame_idx_y = frame_idx_x  # same capture index
    else:
        # two-leg mode
        if streams_y is None:
            raise ValueError("If d_y is provided, streams_y must also be provided.")
        if d_x.ndim != 3 or d_y.ndim != 3:
            raise ValueError("Expected d_x and d_y shape (n_streams, ncap, nsamp)")

    nx_eff = d_x.shape[0]
    ny_eff = d_y.shape[0]

    # ----- infer delays vs ref=0 in each arm -----
    tau_x_ns = estimate_tau_ref_ns(
        d_x, fs, ref_idx=0, frame_idx=frame_idx_x,
        n_segments=n_segments, Nfft=Nfft,
        lag_window=lag_window, refine=refine, up_factor=up_factor
    )
    tau_y_ns = estimate_tau_ref_ns(
        d_y, fs, ref_idx=0, frame_idx=frame_idx_y,
        n_segments=n_segments, Nfft=Nfft,
        lag_window=lag_window, refine=refine, up_factor=up_factor
    )

    x_m = build_effective_geometry_from_tau(tau_x_ns, aperture_m=aperture_x_m, center=True)
    y_m = build_effective_geometry_from_tau(tau_y_ns, aperture_m=aperture_y_m, center=True)

    # sort for display
    if sort_by_pos:
        idx_x = np.argsort(x_m)
        idx_y = np.argsort(y_m)
    else:
        idx_x = np.arange(nx_eff)
        idx_y = np.arange(ny_eff)

    d_x2 = d_x[idx_x, :, :]
    d_y2 = d_y[idx_y, :, :]
    x2 = x_m[idx_x]
    y2 = y_m[idx_y]

    # ----- plot L geometry -----
    Xcoords = np.concatenate([x2, np.zeros_like(y2)])
    Ycoords = np.concatenate([np.zeros_like(x2), y2])
    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
    ax.scatter(Xcoords, Ycoords, s=18)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("East-West x [m]")
    ax.set_ylabel("North-South y [m]")
    ax.set_title("L-array geometry (scaled)")
    ax.grid(True)
    plt.show()

    # ----- beamform each arm -----
    tx, Paz = wideband_delay_and_sum_beam_pattern(
        d_x2, fs, x2,
        frame_idx=frame_idx_x, n_segments=n_segments, Nfft=Nfft,
        theta_deg=theta_x_deg, fmin=fmin, fmax=fmax,
        n_nyqzone=n_nyqzone,        
        show_progress=show_progress, progress_desc="Beam scan (EW)"
    )
    ty, Pel = wideband_delay_and_sum_beam_pattern(
        d_y2, fs, y2,
        frame_idx=frame_idx_y, n_segments=n_segments, Nfft=Nfft,
        theta_deg=theta_y_deg, fmin=fmin, fmax=fmax,
        n_nyqzone=n_nyqzone,
        show_progress=show_progress, progress_desc="Beam scan (NS)"
    )

    u_scan = np.sin(np.deg2rad(tx))
    v_scan = np.sin(np.deg2rad(ty))

    # separable peaks
    iu = int(np.argmax(Paz))
    iv = int(np.argmax(Pel))
    peak_tx = float(tx[iu])
    peak_ty = float(ty[iv])
    peak_u = float(u_scan[iu])
    peak_v = float(v_scan[iv])
    peak_P_uv = float(Paz[iu] * Pel[iv])

    # ----- EW/NS heatmap in θx vs θy -----
    P_uv = Paz[None, :] * Pel[:, None]  # (len(ty), len(tx))
    P_uv_max = float(np.max(P_uv))
    if heatmap_db:
        Z_uv = 10.0 * np.log10(P_uv / P_uv_max)
        peak_uv_db = 10.0 * np.log10(peak_P_uv / P_uv_max)
        cbar_uv = "Power [dB, norm]"
    else:
        Z_uv = P_uv / P_uv_max
        peak_uv_db = 10.0 * np.log10(peak_P_uv / P_uv_max)
        cbar_uv = "Power [linear, norm]"

    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    im = ax.imshow(
        Z_uv, origin="lower", aspect="auto",
        extent=[tx[0], tx[-1], ty[0], ty[-1]],
        interpolation=None,
    )
    ax.scatter([peak_tx], [peak_ty], s=60, color="red", marker="x")
    ax.set_xlabel("EW scan angle θx°")
    ax.set_ylabel("NS scan angle θy°")
    ax.set_title("EW/NS localization")
    fig.colorbar(im, ax=ax, label=cbar_uv)
    ax.text(
        0.02, 0.98,
        ("Peak:\n"
         "  θx = {:.3f}° , θy = {:.3f}°").format(
            peak_tx, peak_ty),
        transform=ax.transAxes, va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
        fontsize=10,
    )
    plt.show()

    # EW/NS slices through peak
    if show_slices:
        fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
        axs[0].plot(tx, Z_uv[iv, :])
        axs[0].axvline(peak_tx, linestyle="--")
        axs[0].set_title("EW/NS slice: along θx at peak θy")
        axs[0].set_xlabel("θx°")
        axs[0].set_ylabel(cbar_uv)
        axs[0].grid(True)

        axs[1].plot(ty, Z_uv[:, iu])
        axs[1].axvline(peak_ty, linestyle="--")
        axs[1].set_title("EW/NS slice: along θy at peak θx")
        axs[1].set_xlabel("θy°")
        axs[1].set_ylabel(cbar_uv)
        axs[1].grid(True)
        plt.show()

    # ----- Spherical AZ/EL heatmap via mapping u,v -----
    az_sph = np.asarray(az_sph_deg, dtype=float)
    el_sph = np.asarray(el_sph_deg, dtype=float)
    AZ, EL = np.meshgrid(az_sph, el_sph, indexing="xy")  # (Nel, Naz)

    azr = np.deg2rad(AZ)
    elr = np.deg2rad(EL)
    u = np.cos(elr) * np.cos(azr)
    v = np.cos(elr) * np.sin(azr)

    def interp1(x, xp, fp):
        return np.interp(x, xp, fp, left=0.0, right=0.0)

    Paz_uv = interp1(u.ravel(), u_scan, Paz).reshape(u.shape)
    Pel_uv = interp1(v.ravel(), v_scan, Pel).reshape(v.shape)
    P_sph = Paz_uv * Pel_uv

    P_sph_max = float(np.max(P_sph))
    peak_flat = int(np.argmax(P_sph))
    peak_el_i, peak_az_i = np.unravel_index(peak_flat, P_sph.shape)
    peak_az = float(az_sph[peak_az_i])
    peak_el = float(el_sph[peak_el_i])
    peak_P_sph = float(P_sph[peak_el_i, peak_az_i])

    if heatmap_db:
        Z_sph = 10.0 * np.log10(P_sph / P_sph_max)
        peak_sph_db = 10.0 * np.log10(peak_P_sph / P_sph_max)
        cbar_sph = "Power [dB, norm]"
    else:
        Z_sph = P_sph / P_sph_max
        peak_sph_db = 10.0 * np.log10(peak_P_sph / P_sph_max)
        cbar_sph = "Power [linear, norm]"

    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    im = ax.imshow(
        Z_sph, origin="lower", aspect="auto",
        extent=[az_sph[0], az_sph[-1], el_sph[0], el_sph[-1]],
        interpolation=None,
    )
    ax.scatter([peak_az], [peak_el], s=60, color="red", marker="x")
    ax.set_xlabel("Azimuth [deg]")
    ax.set_ylabel("Elevation [deg]")
    ax.set_title("Spherical AZ/EL localization (mapped from EW/NS direction cosines)")
    fig.colorbar(im, ax=ax, label=cbar_sph)
    ax.text(
        0.02, 0.1,
        ("Peak:\n"
         "  az = {:.2f}° , el = {:.2f}°").format(peak_az, peak_el),
        transform=ax.transAxes, va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
        fontsize=10,
    )
    plt.show()

    # AZ/EL slices through peak
    if show_slices:
        fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
        axs[0].plot(az_sph, Z_sph[peak_el_i, :])
        axs[0].axvline(peak_az, linestyle="--")
        axs[0].set_title("AZ/EL slice: along az at peak el")
        axs[0].set_xlabel("Az [deg]")
        axs[0].set_ylabel(cbar_sph)
        axs[0].grid(True)

        axs[1].plot(el_sph, Z_sph[:, peak_az_i])
        axs[1].axvline(peak_el, linestyle="--")
        axs[1].set_title("AZ/EL slice: along el at peak az")
        axs[1].set_xlabel("El [deg]")
        axs[1].set_ylabel(cbar_sph)
        axs[1].grid(True)
        plt.show()

    return {
        "x_sorted_m": x2,
        "y_sorted_m": y2,
        "theta_x_deg": tx,
        "theta_y_deg": ty,
        "Paz": Paz,
        "Pel": Pel,
        "u_scan": u_scan,
        "v_scan": v_scan,
        "P_uv": P_uv,
        "P_sph": P_sph,
        "peak_theta_x_deg": peak_tx,
        "peak_theta_y_deg": peak_ty,
        "peak_u": peak_u,
        "peak_v": peak_v,
        "peak_P_uv": peak_P_uv,
        "peak_az_deg": peak_az,
        "peak_el_deg": peak_el,
        "peak_P_sph": peak_P_sph,
    }




# ---------------------------------------------
# 1) Lag spectra products (unique i<j products)
# ---------------------------------------------

def plot_lags_products_unique(
    d, streams, boards=None, chans=None,
    fs=3.2e9, frame_idx=0,

    # segmentation (NON-overlapping by default)
    n_segments=None,      # None or <=1 => full-frame; >1 => split into n_segments non-overlapping blocks
    Nseg=None,            # override segment length; uses as many non-overlapping segments as fit

    # FFT / peak estimation
    Nfft=None,            # auto if None
    refine=True,
    up_factor=16,
    lag_window=(-40, 40),

    # plotting / storage
    hist_bins=120,
    show_hist=True,
    show_heatmap=True,
    show_spectra=False,
    spectra_max=None,     # optionally subsample overplot
    spectra_alpha=0.15,
    heatmap_logscale=False,
):
    """
    Compute and visualize lag spectra for all unique cross-products (i<j) among selected inputs,
    using optional non-overlapping segmentation and cross-spectrum averaging.

    Compared to a Welch estimator, this uses strictly non-overlapping segments by default to
    simplify the interpretation as "independent measurements." Averaging over segments reduces
    variance of the lag spectrum approximately ~1/n_segments_eff.

    Parameters
    ----------
    d : ndarray
        Time-domain data, shape (n_streams, ncap, nsamples). Typical: (64,1,32768).
    streams : sequence
        Stream descriptors corresponding to axis 0 of `d`. Requires `coord_from_stream()`.
    boards, chans : iterable or None
        Optional selection filters on (board, chan/slot).
    fs : float
        Sample rate in Hz.
    frame_idx : int
        Capture index along ncap axis.
    n_segments : int or None
        Desired number of non-overlapping segments. If None or <=1, uses the full frame as one
        segment. If >1, splits the frame into n_segments equal-length non-overlapping blocks
        (rounded down to a multiple of 256 when possible), and uses all blocks that fit.
    Nseg : int or None
        Explicit segment length. If provided, uses as many non-overlapping segments of length Nseg
        as fit in the frame (ignoring n_segments).
    Nfft : int or None
        FFT length for each segment. If None:
          - single segment: Nfft = next_pow2(Nseg)
          - multi segment:  Nfft = next_pow2(2*Nseg)   (reduces circular wrap for small-lag inspection)
    refine : bool
        If True, compute correlation on a finer lag grid by using IFFT length Ncorr=up_factor*Nfft,
        then refine the peak with 3-point parabolic interpolation.
    up_factor : int
        Upsampling factor used when refine=True (typical 8–32).
    lag_window : tuple or None
        Lag search/plot window in sample units (e.g. (-40,40)). None => full axis.

    Returns
    -------
    dict
        Includes lag axis, spectra matrix, peak delays, product ordering, and segmentation metadata.
    """
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamples)")
    n_streams, ncap, nsamp = d.shape
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")

    # select streams
    sel_idx = []
    labels = []
    for i, s in enumerate(streams):
        board, chan = coord_from_stream(s)
        if (boards is None or board in boards) and (chans is None or chan in chans):
            sel_idx.append(i)
            labels.append((board, chan))
    M = len(sel_idx)
    if M < 2:
        raise ValueError("Need at least 2 selected streams.")

    # segmentation (non-overlap)
    Nseg0, starts = _choose_nonoverlap_segments(nsamp, n_segments, Nseg=Nseg)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg0).astype(np.float32)

    # FFT lengths
    Nfft0, up, Ncorr = _auto_fft_lengths(Nseg0, segmented, Nfft=Nfft, up_factor=up_factor, refine=refine)
    lags = (np.arange(Ncorr) - Ncorr // 2) / float(up)

    # lag window indices
    lag_idx, lags_disp = _lag_window_indices(lags, lag_window)

    # compute FFTs
    X = _compute_segment_ffts(d, sel_idx, frame_idx, starts, Nseg0, Nfft0, window)

    # pairs + delays
    pairs = _pair_list(M)
    C_disp, peak_lag_samp, peak_tau_ns, peak_amp = _pair_tau_from_X(
        X, pairs, Ncorr, lags, lag_idx, fs, refine, up
    )

    # sort: primary peak lag, tie-break by -amp
    order = np.lexsort((-peak_amp, peak_lag_samp))

    # histogram
    if show_hist:
        plt.figure(figsize=(8, 4))
        plt.hist(peak_tau_ns, bins=hist_bins)
        plt.xlabel("Peak delay [ns]")
        plt.ylabel("Count (products)")
        plt.title("Peak-delay histogram (unique pairs; P={}, seg={})".format(len(pairs), nseg_eff))
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # spectra overplot (sorted+colored)
    if show_spectra:
        idx_plot = order
        if spectra_max is not None and int(spectra_max) < len(idx_plot):
            rng = np.random.default_rng(0)
            idx_plot = rng.choice(idx_plot, size=int(spectra_max), replace=False)
            idx_plot = idx_plot[np.argsort(peak_lag_samp[idx_plot])]

        vmin, vmax = float(np.min(peak_lag_samp)), float(np.max(peak_lag_samp))
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.cm.coolwarm

        fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
        for p in idx_plot:
            ax.plot(
                lags_disp,
                C_disp[p, :],
                color=cmap(norm(float(peak_lag_samp[p]))),
                alpha=spectra_alpha,
                linewidth=1.0,
            )
        ax.set_xlabel("Lag [samples]")
        ax.set_ylabel("|cross-correlation|")
        ax.set_title("Lag spectra (sorted by peak lag; color = peak lag)")
        ax.grid(True)

        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax)
        cbar.set_label("Peak lag [samples]")
        plt.show()

    # heatmap (sorted rows)
    if show_heatmap:
        C_show = C_disp[order, :]

        if heatmap_logscale:
            pos = C_show[C_show > 0]
            vmin = np.percentile(pos, 1) if pos.size else 1e-12
            vmax = np.percentile(C_show, 99)
            norm = LogNorm(vmin=max(vmin, 1e-12), vmax=max(vmax, max(vmin, 1e-12) * 1.01))
            cbar_label = "|r| (log)"
        else:
            norm = None
            cbar_label = "|r|"

        fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True)
        im = ax.imshow(
            C_show,
            aspect="auto",
            origin="lower",
            extent=[lags_disp[0], lags_disp[-1], 0, len(pairs)],
            interpolation="nearest",
            norm=norm,
        )
        ax.set_xlabel("Lag [samples]")
        ax.set_ylabel("Product index (sorted by peak lag)")
        ax.set_title("|cross-correlation| heatmap (P={}, seg={}, refine={}, up={})".format(
            len(pairs), nseg_eff, refine, up
        ))
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label(cbar_label)
        plt.show()

    return {
        "lags_disp_samples": lags_disp,
        "corr_mag_disp": C_disp,          # unsorted product order
        "order": order,                   # sort order by peak lag
        "peak_tau_ns": peak_tau_ns,
        "peak_lag_samples": peak_lag_samp,
        "peak_amp": peak_amp,
        "prod_ij": pairs,                 # in selected-stream index space
        "labels": labels,
        "sel_idx": sel_idx,
        "segmentation": {
            "nsamp": int(nsamp),
            "Nseg": int(Nseg0),
            "starts": starts,
            "n_segments_effective": int(nseg_eff),
            "Nfft": int(Nfft0),
            "Ncorr": int(Ncorr),
            "refine": bool(refine),
            "up": int(up),
        },
    }


# -------------------------------------------------------
# 2) Closure (phase-closure-style) histogram from lag peaks
# -------------------------------------------------------

def closure_histogram_from_lag_peaks(
    d, streams,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,

    # segmentation (NON-overlapping by default)
    n_segments=None,
    Nseg=None,

    # FFT / peak estimation
    Nfft=None,
    refine=True,
    up_factor=16,
    lag_window=(-75, 75),

    # output/plot
    hist_bins=200,
    show_debug=False,
):
    """
    Compute closure (self-consistency) residuals of pairwise lag-spectrum delay estimates
    using a single reference channel (stream 0 in the *selected* set).

    This is the delay-domain analogue of phase closure in interferometry: from the set of
    measured pairwise delays τ̂_ij, we test whether they are transitive, i.e.
        τ̂_ij ≈ τ̂_0j − τ̂_0i
    for all i,j != 0.

    Segmentation
    ------------
    Uses the same non-overlapping segmentation scheme as plot_lags_products_unique:
      - If n_segments is None or <=1: single full-frame segment.
      - If n_segments > 1: split into n_segments equal non-overlapping blocks (as many as fit).
      - Alternatively specify Nseg explicitly.

    Method
    ------
    1) For each segment, compute FFTs of each stream.
    2) For each pair (i<j), average cross-spectra across segments:
           Ḡ_ij(f) = mean_m X_i^(m)(f) * conj(X_j^(m)(f))
    3) Compute lag spectrum via IFFT and pick the peak (optional upsampling + parabola).
    4) Use stream 0 as reference: build τ̂_0i for all i.
    5) For each i<j with i,j != 0, compute closure residual:
           Δ_ij = τ̂_ij − (τ̂_0j − τ̂_0i)

    Parameters
    ----------
    d : ndarray
        Time-domain data, shape (n_streams, ncap, nsamples).
    streams : sequence
        Stream descriptors corresponding to axis 0 of d (coord_from_stream used for debug only).
    boards, chans : iterable or None
        Optional selection filters. If provided, closure is computed on the selected subset.
        Reference is the first selected stream (index 0 within the selected list).
    fs : float
        Sampling rate in Hz.
    frame_idx : int
        Capture index along ncap axis.
    n_segments, Nseg, Nfft, refine, up_factor, lag_window : see above.
    hist_bins : int
        Histogram bins for residuals.
    show_debug : bool
        Print basic diagnostic stats and counts.

    Returns
    -------
    dict
        residuals_ns : ndarray
            Closure residuals Δ_ij in ns, length C(N-1,2) for N selected streams.
        tau0_ns : ndarray
            Reference delays τ̂_0i in ns for selected streams (tau0_ns[0]=0).
        segmentation : dict
            Segmenting/FFT metadata.
    """
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamples)")
    n_streams, ncap, nsamp = d.shape
    if frame_idx < 0 or frame_idx >= ncap:
        raise ValueError("frame_idx out of range")

    # select streams
    sel_idx = []
    sel_labels = []
    for i, s in enumerate(streams):
        board, chan = coord_from_stream(s)
        if (boards is None or board in boards) and (chans is None or chan in chans):
            sel_idx.append(i)
            sel_labels.append((board, chan))
    M = len(sel_idx)
    if M < 2:
        raise ValueError("Need at least 2 selected streams.")

    # segmentation (non-overlap)
    Nseg0, starts = _choose_nonoverlap_segments(nsamp, n_segments, Nseg=Nseg)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    window = np.hanning(Nseg0).astype(np.float32)

    # FFT/correlation lengths
    Nfft0, up, Ncorr = _auto_fft_lengths(Nseg0, segmented, Nfft=Nfft, up_factor=up_factor, refine=refine)
    lags = (np.arange(Ncorr) - Ncorr // 2) / float(up)

    # lag window
    lag_idx, _lags_disp = _lag_window_indices(lags, lag_window)

    # FFTs for selected streams
    X = _compute_segment_ffts(d, sel_idx, frame_idx, starts, Nseg0, Nfft0, window)

    # compute tau for all pairs and store antisymmetrically
    tau_ij_ns = {}

    pairs = _pair_list(M)
    for (i, j) in pairs:
        Gbar = np.mean(X[i, :, :] * np.conjugate(X[j, :, :]), axis=0)
        r = np.fft.ifft(Gbar, n=Ncorr)
        mag = np.abs(np.fft.fftshift(r))

        mag_w = mag[lag_idx]
        kw = int(np.argmax(mag_w))
        k_full = int(kw if isinstance(lag_idx, slice) else lag_idx[kw])

        if refine and (0 < k_full < len(mag) - 1):
            delta = parabolic_subbin(mag, k_full)
            lag_hat = lags[k_full] + delta / float(up)
        else:
            lag_hat = lags[k_full]

        tau_hat_ns = (lag_hat / fs) * 1e9
        tau_ij_ns[(i, j)] = tau_hat_ns
        tau_ij_ns[(j, i)] = -tau_hat_ns

    # reference delays tau0[i] = tau(0,i)
    tau0 = np.zeros(M, dtype=np.float64)
    for i in range(1, M):
        tau0[i] = tau_ij_ns[(0, i)]

    # residuals for i<j, i,j != 0
    residuals = []
    for i in range(1, M):
        for j in range(i + 1, M):
            tau_meas = tau_ij_ns[(i, j)]
            tau_pred = tau0[j] - tau0[i]
            residuals.append(tau_meas - tau_pred)
    residuals = np.asarray(residuals, dtype=np.float64)

    expected = (M - 1) * (M - 2) // 2
    if show_debug:
        b0, s0 = sel_labels[0]
        print("Reference (selected index 0): board {}, slot/chan {}".format(b0, s0))
        print("Selected streams: {}".format(M))
        print("Segmentation: nsamp={}, Nseg={}, nseg_eff={}, Nfft={}, up={}".format(
            nsamp, Nseg0, nseg_eff, Nfft0, up
        ))
        print("Residual count: {} (expected {})".format(len(residuals), expected))
        if len(residuals) > 1:
            print("Residual mean {:.6g} ns, std {:.6g} ns".format(
                float(residuals.mean()), float(residuals.std(ddof=1))
            ))

    # plot histogram
    plt.figure(figsize=(8, 4))
    plt.hist(residuals, bins=hist_bins)
    plt.xlabel("Closure residual [ns]")
    plt.ylabel("Count")
    plt.title("Closure residuals (ref=0 in selected set; N={}, seg={}, refine={})".format(
        M, nseg_eff, refine
    ))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return {
        "residuals_ns": residuals,
        "tau0_ns": tau0,
        "n_streams_selected": M,
        "sel_idx": sel_idx,
        "labels_selected": sel_labels,
        "refine": bool(refine),
        "up": int(up),
        "lag_window": lag_window,
        "segmentation": {
            "nsamp": int(nsamp),
            "Nseg": int(Nseg0),
            "starts": starts,
            "n_segments_effective": int(nseg_eff),
            "Nfft": int(Nfft0),
            "Ncorr": int(Ncorr),
        },
    }


# ---------------------------------------------------
# 3) Convenience wrapper to run both analyses together
# ---------------------------------------------------

def lag_spectra_and_closure_report(
    d, streams,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=None,
    Nseg=None,
    Nfft=None,
    refine=True,
    up_factor=16,
    lag_window_spectra=(-40, 40),
    lag_window_closure=(-200, 200),
    hist_bins_spectra=120,
    hist_bins_closure=200,
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.15,
    heatmap_logscale=False,
    show_debug=False,
):
    """
    Convenience function to generate:
      (a) lag spectra histogram + heatmap (sorted by peak lag)
      (b) closure residual histogram

    using the *same* segmentation/FFT/peak-estimation settings.

    Returns a dict with keys:
      - "spectra": output of plot_lags_products_unique(...)
      - "closure": output of closure_histogram_from_lag_peaks(...)
    """
    spectra_res = plot_lags_products_unique(
        d, streams, boards=boards, chans=chans,
        fs=fs, frame_idx=frame_idx,
        n_segments=n_segments, Nseg=Nseg,
        Nfft=Nfft, refine=refine, up_factor=up_factor,
        lag_window=lag_window_spectra,
        hist_bins=hist_bins_spectra,
        show_hist=True,
        show_heatmap=show_heatmap,
        show_spectra=show_spectra,
        spectra_max=spectra_max,
        spectra_alpha=spectra_alpha,
        heatmap_logscale=heatmap_logscale,
    )

    closure_res = closure_histogram_from_lag_peaks(
        d, streams, boards=boards, chans=chans,
        fs=fs, frame_idx=frame_idx,
        n_segments=n_segments, Nseg=Nseg,
        Nfft=Nfft, refine=refine, up_factor=up_factor,
        lag_window=lag_window_closure,
        hist_bins=hist_bins_closure,
        show_debug=show_debug,
    )

    return {"spectra": spectra_res, "closure": closure_res}

# =============================================================================
# 1) Correlation vs reference channel -> lag axis + r_i0(tau)
# =============================================================================

def corr_vs_reference(
    d,
    fs,
    ref_idx=0,
    frame_idx=0,
    n_segments=None,
    Nseg=None,
    Nfft=None,
    up_factor=8,
    lag_window_s=(0.0, 200e-9),
    detrend_mean=True,
    window="hann",
):
    """
    Compute complex cross-correlation r_i0(tau) for all channels i against reference ref_idx.

    Returns:
      tau_s : (L,) seconds
      R     : (n_streams, L) complex64, where R[i,:] = r_i0(tau) in the selected lag window.
    """
    if d.ndim != 3:
        raise ValueError("Expected d shape (n_streams, ncap, nsamp)")
    n_streams, ncap, nsamp = d.shape
    if not (0 <= frame_idx < ncap):
        raise ValueError("frame_idx out of range")
    if not (0 <= ref_idx < n_streams):
        raise ValueError("ref_idx out of range")

    Nseg0, starts = _choose_nonoverlap_segments(nsamp, n_segments=n_segments, Nseg=Nseg)
    nseg_eff = int(starts.size)
    segmented = (nseg_eff > 1)

    if window == "hann":
        w = np.hanning(Nseg0).astype(np.float32)
    else:
        raise ValueError("Only window='hann' supported")

    Nfft0, up, Ncorr = _auto_fft_lengths(Nseg0, segmented, Nfft=Nfft, up_factor=up_factor)

    lags_s = (np.arange(Ncorr) - Ncorr // 2) / float(up) / fs
    lag_idx, tau_s = _lag_window_indices(lags_s, lag_window_s)

    X = np.empty((n_streams, nseg_eff, Nfft0 // 2 + 1), dtype=np.complex64)
    for i in range(n_streams):
        xfull = d[i, frame_idx, :]
        for m, st in enumerate(starts):
            seg = xfull[st:st + Nseg0]
            if detrend_mean:
                seg = seg - np.mean(seg)
            segw = seg * w
            X[i, m, :] = np.fft.rfft(segw, n=Nfft0)

    R = np.empty((n_streams, tau_s.size if not isinstance(lag_idx, slice) else len(lags_s)), dtype=np.complex64)
    Xrefc = np.conjugate(X[ref_idx, :, :])  # (nseg, nfreq)
    for i in range(n_streams):
        Gbar = np.mean(X[i, :, :] * Xrefc, axis=0)
        r = np.fft.ifft(Gbar, n=Ncorr)
        r = np.fft.fftshift(r)
        R[i, :] = r[lag_idx].astype(np.complex64, copy=False)

    meta = {
        "nsamp": int(nsamp),
        "Nseg": int(Nseg0),
        "n_segments_effective": int(nseg_eff),
        "Nfft": int(Nfft0),
        "Ncorr": int(Ncorr),
        "up_factor": int(up),
        "lag_window_s": lag_window_s,
        "ref_idx": int(ref_idx),
    }
    return tau_s.astype(np.float64), R, meta



# ----------------------------
# Geometry: L layout
# ----------------------------
def make_rx_positions_L(n_ch=64, nx=32, ny=32, x_span_m=5.0, y_span_m=5.0, z_spread_m=0.05, seed=0):
    """
    Returns rx_pos_m shape (n_ch,3) with:
      ch0 at origin (Tx/ref)
      ch1..nx on x-arm (y=0)
      next ny on y-arm (x=0)
    """
    if 1 + nx + ny > n_ch:
        raise ValueError("Need n_ch >= 1 + nx + ny")

    rng = np.random.default_rng(seed)
    rx = np.zeros((n_ch, 3), dtype=float)

    # ch0: Tx/ref at origin (you can change if you want)
    rx[0] = np.array([0.0, 0.0, 0.0])

    x = np.linspace(-x_span_m/2, x_span_m/2, nx)
    y = np.linspace(-y_span_m/2, y_span_m/2, ny)

    rx[1:1+nx, 0] = x
    rx[1:1+nx, 1] = 0.0
    rx[1:1+nx, 2] = z_spread_m * rng.normal(size=nx)

    start = 1 + nx
    rx[start:start+ny, 0] = 0.0
    rx[start:start+ny, 1] = y
    rx[start:start+ny, 2] = z_spread_m * rng.normal(size=ny)

    return rx


def pick_3_receivers_farthest(rx_pos_m, candidate_indices):
    """
    Pick 3 farthest-apart receivers from candidate_indices (good conditioning).
    """
    cand = np.asarray(candidate_indices, dtype=int)
    P = np.asarray(rx_pos_m, dtype=float)[cand]
    if P.shape[0] < 3:
        raise ValueError("Need at least 3 candidates")

    D = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=2)
    i, j = np.unravel_index(np.argmax(D), D.shape)
    di, dj = D[i], D[j]
    k = int(np.argmax(np.minimum(di, dj)))
    return [int(cand[i]), int(cand[j]), int(cand[k])]


# ----------------------------
# Lag-spectrum sampling helpers
# ----------------------------
def _interp_complex_1d(xp, fp, xq):
    re = np.interp(xq, xp, fp.real, left=0.0, right=0.0)
    im = np.interp(xq, xp, fp.imag, left=0.0, right=0.0)
    return re + 1j * im


def estimate_tau0_monostatic_from_peaks(tau_s, R, rx_pos_m, idx3, p_guess_m):
    """
    tau_peak_i ≈ tau0 + |p_guess - r_i|/c  =>  tau0 = mean(tau_peak_i - |p-r_i|/c)
    """
    p = np.asarray(p_guess_m, float).reshape(3)
    tau0_list = []
    for i in idx3:
        mag = np.abs(R[i])
        k = int(np.argmax(mag))
        tau_peak = float(tau_s[k])
        rho = float(np.linalg.norm(p - np.asarray(rx_pos_m[i], float)))
        tau0_list.append(tau_peak - rho / C0)
    return float(np.mean(tau0_list))


def monostatic_range_az_el_score_3rx(
    tau_s,
    R,
    rx_pos_m,
    idx3,
    tau0_s,
    range_m_grid,
    az_deg_grid,
    el_deg_grid,
    origin_m=np.array([0.0, 0.0, 0.0]),
    use_magnitude_sum=True,
):
    """
    Build Sdb(R, el, az) for monostatic model using 3 receivers:
      p = origin + R * d(az,el)
      tau_i(p) = tau0 + |p-r_i|/c
      score = sum_i |r_i0(tau_i)|  (robust) or |sum r_i0(tau_i)| (coherent)
    """
    tau_s = np.asarray(tau_s, float)
    rx = np.asarray(rx_pos_m, float)
    idx3 = np.asarray(idx3, int)

    Rm = np.asarray(range_m_grid, float)
    az = np.deg2rad(np.asarray(az_deg_grid, float))
    el = np.deg2rad(np.asarray(el_deg_grid, float))
    Nr, Naz, Nel = Rm.size, az.size, el.size

    origin = np.asarray(origin_m, float).reshape(3)

    Ri = [R[int(i)] for i in idx3]

    S = np.zeros((Nr, Nel, Naz), dtype=float)

    for ie, elr in enumerate(el):
        cosel = np.cos(elr)
        sinel = np.sin(elr)
        dirs = np.stack([cosel*np.cos(az), cosel*np.sin(az), np.full(Naz, sinel)], axis=1)  # (Naz,3)

        for ir, Rval in enumerate(Rm):
            p = origin[None, :] + Rval * dirs  # (Naz,3)

            score = np.zeros(Naz, dtype=np.float64 if use_magnitude_sum else np.complex128)

            for k, ch in enumerate(idx3):
                rho = np.linalg.norm(p - rx[ch][None, :], axis=1)
                tau_pred = float(tau0_s) + rho / C0
                samp = _interp_complex_1d(tau_s, Ri[k], tau_pred)

                if use_magnitude_sum:
                    score += np.abs(samp)
                else:
                    score += samp

            if use_magnitude_sum:
                S[ir, ie, :] = score**2
            else:
                S[ir, ie, :] = (score.real**2 + score.imag**2)

    Smax = float(np.max(S)) if np.max(S) > 0 else 1.0
    Sdb = 10.0 * np.log10(S / Smax + 1e-300)
    return Sdb


# ----------------------------
# Visualization: global + local sphere patches (true surfaces)
# ----------------------------
def monostatic_tau_pred(tau0_s, p_xyz, rx_xyz):
    return float(tau0_s) + float(np.linalg.norm(np.asarray(p_xyz) - np.asarray(rx_xyz))) / C0


def local_peak_near(tau_s, mag, tau0, search_halfwidth_ns=3.0):
    hw = float(search_halfwidth_ns) * 1e-9
    lo, hi = float(tau0) - hw, float(tau0) + hw
    m = (tau_s >= lo) & (tau_s <= hi)
    if not np.any(m):
        j = int(np.argmin(np.abs(tau_s - tau0)))
        return float(tau_s[j]), float(mag[j])
    idx = np.where(m)[0]
    jloc = idx[int(np.argmax(mag[idx]))]
    return float(tau_s[jloc]), float(mag[jloc])


def find_best_point_in_Sdb(Sdb, range_m_grid, az_deg_grid, el_deg_grid, origin_m):
    Nr, Nel, Naz = Sdb.shape
    flat = int(np.argmax(Sdb))
    ir, ie, ia = np.unravel_index(flat, (Nr, Nel, Naz))

    Rm = float(range_m_grid[ir])
    az = np.deg2rad(float(az_deg_grid[ia]))
    el = np.deg2rad(float(el_deg_grid[ie]))

    d = np.array([np.cos(el)*np.cos(az), np.cos(el)*np.sin(az), np.sin(el)], dtype=float)
    origin = np.asarray(origin_m, float).reshape(3)
    p = origin + Rm * d

    return {
        "ir": int(ir), "ie": int(ie), "ia": int(ia),
        "R_m": Rm,
        "az_deg": float(az_deg_grid[ia]),
        "el_deg": float(el_deg_grid[ie]),
        "p_xyz": p,
        "score_db": float(Sdb[ir, ie, ia]),
    }


def _orthonormal_basis(u):
    u = np.asarray(u, float)
    u = u / (np.linalg.norm(u) + 1e-300)
    a = np.array([1.0, 0.0, 0.0], float)
    if abs(np.dot(a, u)) > 0.9:
        a = np.array([0.0, 1.0, 0.0], float)
    e1 = np.cross(u, a); e1 /= (np.linalg.norm(e1) + 1e-300)
    e2 = np.cross(u, e1); e2 /= (np.linalg.norm(e2) + 1e-300)
    return u, e1, e2


def sphere_cap_surface(center, radius, toward_point, cap_half_angle_deg=12.0, n_theta=45, n_phi=80):
    c = np.asarray(center, float).reshape(3)
    r = float(radius)
    p = np.asarray(toward_point, float).reshape(3)

    u = p - c
    nu = np.linalg.norm(u)
    if nu <= 0:
        u = np.array([0.0, 0.0, 1.0])
    else:
        u = u / nu

    u, e1, e2 = _orthonormal_basis(u)
    amax = np.deg2rad(float(cap_half_angle_deg))

    a = np.linspace(0.0, amax, n_theta)
    b = np.linspace(0.0, 2*np.pi, n_phi)
    A, B = np.meshgrid(a, b, indexing="xy")  # (n_phi, n_theta)

    dir_vec = (
        np.cos(A)[..., None] * u[None, None, :] +
        np.sin(A)[..., None] * (
            np.cos(B)[..., None] * e1[None, None, :] +
            np.sin(B)[..., None] * e2[None, None, :]
        )
    )
    pts = c[None, None, :] + r * dir_vec
    return pts[..., 0], pts[..., 1], pts[..., 2]


def plot_global_L_with_local_sphere_patches(
    tau_s,
    R_corr,
    rx_pos_m,
    idx3,
    tau0_s,
    Sdb,
    range_m_grid,
    az_deg_grid,
    el_deg_grid,
    origin_m=np.array([0.0, 0.0, 0.0]),
    cap_half_angle_deg=12.0,
    cap_n_theta=45,
    cap_n_phi=80,
    peak_search_halfwidth_ns=3.0,
    title_prefix="",
):
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    tau_s = np.asarray(tau_s, float)
    rx = np.asarray(rx_pos_m, float)
    idx3 = list(map(int, idx3))

    best = find_best_point_in_Sdb(Sdb, range_m_grid, az_deg_grid, el_deg_grid, origin_m)
    p_star = best["p_xyz"]

    # radii from local peaks near predicted delay at p*
    radii = []
    resid_ns = []
    for ch in idx3:
        tau_pred = monostatic_tau_pred(tau0_s, p_star, rx[ch])
        mag = np.abs(R_corr[ch, :])
        tau_hat, _ = local_peak_near(tau_s, mag, tau_pred, search_halfwidth_ns=peak_search_halfwidth_ns)
        resid_ns.append((tau_hat - tau_pred) * 1e9)
        radii.append(max(0.0, C0 * (tau_hat - float(tau0_s))))
    radii = np.asarray(radii, float)

    fig = plt.figure(figsize=(9, 8.5))
    ax = fig.add_subplot(111, projection="3d")

    # plot array as L (because rx_pos_m actually *is* L)
    ax.scatter(rx[:, 0], rx[:, 1], rx[:, 2], s=12, alpha=0.22, label="Array (L)")

    # Tx/ref at ch0
    ax.scatter([rx[0, 0]], [rx[0, 1]], [rx[0, 2]], s=130, marker="^", label="Monostatic Reference")

    # chosen receivers
    sel = rx[idx3, :]
    ax.scatter(sel[:, 0], sel[:, 1], sel[:, 2], s=120, alpha=0.95, label=f"Receiver Subset")

    # p*
    ax.scatter([p_star[0]], [p_star[1]], [p_star[2]], s=180, marker="x", label="Target")

    # local sphere patches (true surfaces)
    for c, r in zip(sel, radii):
        if r <= 0:
            continue
        X, Y, Z = sphere_cap_surface(
            center=c,
            radius=r,
            toward_point=p_star,
            cap_half_angle_deg=cap_half_angle_deg,
            n_theta=cap_n_theta,
            n_phi=cap_n_phi,
        )
        ax.plot_surface(X, Y, Z, linewidth=0, antialiased=True, alpha=0.35)

    txt = (
        f"Target={best['R_m']:.2f} m, az={best['az_deg']:.1f}°, el={best['el_deg']:.1f}°")
    ax.text(p_star[0], p_star[1], p_star[2], "  " + txt, fontsize=9)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title(f"{title_prefix}L-array with active monostatic localization")
    ax.legend(loc="upper left")

    # bounds: show whole L and p*
    mins = np.minimum(rx.min(axis=0), p_star)
    maxs = np.maximum(rx.max(axis=0), p_star)
    span = max(maxs - mins)
    mid = 0.5 * (mins + maxs)
    ax.set_xlim(mid[0] - 0.55*span, mid[0] + 0.55*span)
    ax.set_ylim(mid[1] - 0.55*span, mid[1] + 0.55*span)
    ax.set_zlim(mid[2] - 0.55*span, mid[2] + 0.55*span)

    plt.tight_layout()
    plt.show()

    return {"p_star": p_star, "best": best, "radii_m": radii, "resid_ns": np.asarray(resid_ns)}

def plot_range_vs_az_el_maps_from_Sdb(
    Sdb,
    range_m_grid,
    az_deg_grid,
    el_deg_grid,
    el_slice_deg=None,
    az_slice_deg=None,
    dyn_db=20.0,
    title_prefix="",
    show_range_profile=True,
    # --- heatmap-only range extent control ---
    rmin_m=None,
    rmax_m=None,
    r_extent_m=None,   # optional tuple (rmin, rmax) overrides rmin_m/rmax_m if provided
    # --- range profile units ---
    range_profile_units="linear_power",  # "db", "linear_power", "linear_amplitude"
):
    """
    Sdb shape (Nr, Nel, Naz) in dB-rel (0 dB = global max).

    - Range profile (optional) uses FULL range (ignores rmin/rmax).
    - Heatmaps (Range–Az, Range–El) can be range-cropped via rmin/rmax (heatmap-only).
    """
    Sdb = np.asarray(Sdb, dtype=float)
    Rm = np.asarray(range_m_grid, dtype=float)
    az = np.asarray(az_deg_grid, dtype=float)
    el = np.asarray(el_deg_grid, dtype=float)

    if Sdb.ndim != 3:
        raise ValueError("Sdb must have shape (Nr, Nel, Naz)")

    Nr, Nel, Naz = Sdb.shape
    if (Nr != Rm.size) or (Nel != el.size) or (Naz != az.size):
        raise ValueError("Sdb shape must match provided grids")

    # -------------------------------------------------------------------------
    # Global maximum (for default slice selection + markers)
    # -------------------------------------------------------------------------
    flat = int(np.argmax(Sdb))
    ir_star, ie_star, ia_star = np.unravel_index(flat, Sdb.shape)

    # Choose slice indices
    if el_slice_deg is None:
        ie = int(ie_star)
    else:
        ie = int(np.argmin(np.abs(el - float(el_slice_deg))))

    if az_slice_deg is None:
        ia = int(ia_star)
    else:
        ia = int(np.argmin(np.abs(az - float(az_slice_deg))))

    vmin, vmax = -float(dyn_db), 0.0

    # -------------------------------------------------------------------------
    # Optional range profile (FULL range, ignores rmin/rmax)
    # -------------------------------------------------------------------------
    if show_range_profile:
        prof_db = np.max(Sdb, axis=(1, 2))  # (Nr,) dB-rel (full range)

        u = range_profile_units.lower()
        if u == "db":
            prof_y = prof_db
            ylab = "Max score [dB rel]"
        elif u == "linear_power":
            prof_y = 10.0 ** (prof_db / 10.0)   # power-linear, peak=1
            ylab = "Max score [linear power, rel]"
        elif u == "linear_amplitude":
            prof_y = 10.0 ** (prof_db / 20.0)   # amplitude-linear, peak=1
            ylab = "Max score [linear amplitude, rel]"
        else:
            raise ValueError("range_profile_units must be one of: 'db', 'linear_power', 'linear_amplitude'")

        plt.figure(figsize=(9, 3.4))
        plt.plot(Rm, prof_y)
        plt.axvline(Rm[ir_star], linestyle="--")
        plt.xlabel("Range [m] (ray from origin)")
        plt.ylabel(ylab)
        plt.title(f"{title_prefix}Range profile (max over az/el) | peak @ {Rm[ir_star]:.2f} m")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # -------------------------------------------------------------------------
    # Heatmap-only range extent selection (crop indices)
    # -------------------------------------------------------------------------
    if r_extent_m is not None:
        rmin_m, rmax_m = float(r_extent_m[0]), float(r_extent_m[1])

    if rmin_m is None:
        rmin_m = float(Rm[0])
    if rmax_m is None:
        rmax_m = float(Rm[-1])

    if rmax_m < rmin_m:
        raise ValueError("Range extent invalid: rmax_m < rmin_m")

    mR = (Rm >= rmin_m) & (Rm <= rmax_m)
    if not np.any(mR):
        raise ValueError("No range samples inside requested heatmap extent.")
    ir_lo = int(np.argmax(mR))                      # first True
    ir_hi = int(len(mR) - 1 - np.argmax(mR[::-1]))  # last True

    Rm_c = Rm[ir_lo:ir_hi + 1]
    Sdb_c = Sdb[ir_lo:ir_hi + 1, :, :]  # crop range dimension for heatmaps

    # peak marker visibility within heatmap extent
    peak_in_extent = (ir_lo <= ir_star <= ir_hi)

    # Extract slices (cropped)
    RA = Sdb_c[:, ie, :]     # (Nr_c, Naz)  Range–Az @ fixed el
    RE = Sdb_c[:, :, ia]     # (Nr_c, Nel)  Range–El @ fixed az

    # -------------------------------------------------------------------------
    # Range–Az heatmap (cropped)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(10, 4.6))
    im = plt.imshow(
        np.maximum(RA, vmin),
        origin="lower",
        aspect="auto",
        extent=[az[0], az[-1], Rm_c[0], Rm_c[-1]],
        interpolation=None,
        vmin=vmin,
        vmax=vmax,
    )
    plt.xlabel("Azimuth [deg]")
    plt.ylabel("Range [m] (ray from origin)")
    plt.title(f"{title_prefix}Range–Az @ el={el[ie]:.2f}° (slice)")
    plt.colorbar(im, label=f"Score [dB rel, shown {vmin:.0f}..0]")

    if peak_in_extent:
        plt.scatter([az[ia_star]], [Rm[ir_star]], s=40, marker="x", color="red")

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # Range–El heatmap (cropped)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(10, 4.6))
    im = plt.imshow(
        np.maximum(RE, vmin),
        origin="lower",
        aspect="auto",
        extent=[el[0], el[-1], Rm_c[0], Rm_c[-1]],
        interpolation=None,
        vmin=vmin,
        vmax=vmax,
    )
    plt.xlabel("Elevation [deg]")
    plt.ylabel("Range [m] (ray from origin)")
    plt.title(f"{title_prefix}Range–El @ az={az[ia]:.2f}° (slice)")
    plt.colorbar(im, label=f"Score [dB rel, shown {vmin:.0f}..0]")

    if peak_in_extent:
        plt.scatter([el[ie_star]], [Rm[ir_star]], s=40, marker="x", color="red")

    plt.tight_layout()
    plt.show()

    return {
        "ir_star": int(ir_star),
        "ie_star": int(ie_star),
        "ia_star": int(ia_star),
        "range_star_m": float(Rm[ir_star]),
        "az_star_deg": float(az[ia_star]),
        "el_star_deg": float(el[ie_star]),
        "el_slice_deg_used": float(el[ie]),
        "az_slice_deg_used": float(az[ia]),
        "heatmap_range_extent_used_m": (float(Rm_c[0]), float(Rm_c[-1])),
        "peak_in_heatmap_extent": bool(peak_in_extent),
    }



# Load Data

In [ ]:
dat = pkl.load(open('corr_gain_cal_fft_data_newf_1.pkl','rb'))
dd0_all_n1 = dat['d']
gain_theory_n1 = dat['gain']
prod_map_n1 = dat['prod_map']

dat = pkl.load(open('corr_gain_cal_fft_data_newf_2.pkl','rb'))
dd0_all_n2 = dat['d']
gain_theory_n2 = dat['gain']
prod_map_n2 = dat['prod_map']

dat = pkl.load(open('corr_gain_cal_fft_data_newf_3.pkl','rb'))
dd0_all_n3 = dat['d']
gain_theory_n3 = dat['gain']
prod_map_n3 = dat['prod_map']


In [ ]:
dat_adc = pkl.load(open('corr_adc_data_newf_1.pkl','rb'))
d_adc_n1 = dat_adc['d']
streams_n1 = dat_adc['streams'] 

dat_adc = pkl.load(open('corr_adc_data_newf_2.pkl','rb'))
d_adc_n2 = dat_adc['d']
streams_n2 = dat_adc['streams'] 

dat_adc = pkl.load(open('corr_adc_data_newf_3.pkl','rb'))
d_adc_n3 = dat_adc['d']
streams_n3 = dat_adc['streams'] 


In [ ]:
M = 2**14 # samples
fs = 3200 # MHz
f_res = fs / M
freq_n1 = f_res*np.arange(M//2)#[::-1]
freq_n2 = fs - f_res*np.arange(M//2)#[::-1]
freq_n3 = fs + f_res*np.arange(M//2)#[::-1]
freq_n4 = 2*fs - f_res*np.arange(M//2)#[::-1]


Ts = 1/(fs*1e6) # sample period in seconds
t_frame = Ts*np.arange(M) # frame duration in seconds

In [ ]:
fac_ = 1 / 2**14 # Vpp / V_lsb  

plt.figure()
plt.plot(10* np.log10(np.std(d_adc_n1[:,0,:] * fac_,axis=-1)**2 / 100) + 30,'-o' ,label='New Firm. 1')
plt.plot(10* np.log10(np.std(d_adc_n2[:,0,:] * fac_,axis=-1)**2 / 100) + 30,'-o' ,label='New Firm. 2')
plt.plot(10* np.log10(np.std(d_adc_n3[:,0,:] * fac_,axis=-1)**2 / 100) + 30,'-o' ,label='New Firm. 3')
plt.legend(loc=0)
plt.ylabel('Power [dBm]')
plt.xlabel('Correlator Input')
plt.show()


In [ ]:
plt.figure()
plt.plot( (d_adc_n1[2,0]) ,label='1')
plt.plot( (d_adc_n2[2,0]) ,label='2')
plt.plot( (d_adc_n4[2,0]) ,label='4')
plt.plot( (d_adc_n3[2,0]) ,label='3')

#plt.plot( (d_adc_n2[0,0]))
#plt.plot( (d_adc_n3[0,0]))
#plt.plot( (d_adc_n4[0,0]))
#plt.plot(d_adc_n2[0,0])
#plt.plot(d_adc_n3[0,0])

#plt.plot(d_adc_n4[0,0])
plt.legend(loc=0)
plt.show()

plt.figure()
plt.plot( np.abs(np.fft.rfft(d_adc_n1[2,0])) ,label='1')
plt.plot( np.abs(np.fft.rfft(d_adc_n2[2,0])) ,label='2')
plt.plot( np.abs(np.fft.rfft(d_adc_n3[2,0])) ,label='3')
plt.plot( np.abs(np.fft.rfft(d_adc_n4[2,0])) ,label='4')
#plt.plot( (d_adc_n2[0,0]) ,label='2')
#plt.plot( (d_adc_n4[0,0]) ,label='4')
#plt.plot( (d_adc_n3[0,0]) ,label='3')

#plt.plot( (d_adc_n2[0,0]))
#plt.plot( (d_adc_n3[0,0]))
#plt.plot( (d_adc_n4[0,0]))
#plt.plot(d_adc_n2[0,0])
#plt.plot(d_adc_n3[0,0])

#plt.plot(d_adc_n4[0,0])
plt.yscale('log')
plt.legend(loc=0)
plt.show()



# Lag Spectras

## Correlator Products

In [ ]:
#  [prod_map[i, j] for i in range(32) for j in range(i + 1,8)]
prod_inds = [prod_map_n1[i, j] for i in range(64) for j in range(i + 1,64)]

#print(cr.NCHAN)

In [ ]:

dd0_all_nogain_n1 = np.zeros_like(dd0_all_n1)
dd0_all_nogain_n2 = np.zeros_like(dd0_all_n2)
dd0_all_nogain_n3 = np.zeros_like(dd0_all_n3)
#dd0_all_nogain_n4 = np.zeros_like(dd0_all_n4)

for prodii in range(dd0_all_n1.shape[1]):
    prodi, prodj = np.where(prod_map_n1 == prodii)
    dd0_all_nogain_n1[:,prodii] = dd0_all_n1[:,prodii] * (2**44)**2 \
                                /gain_theory_n1[prodi[0]]\
                                /gain_theory_n1[prodj[0]]\
                                /16384

    dd0_all_nogain_n2[:,prodii] = dd0_all_n2[:,prodii] * (2**44)**2 \
                                /gain_theory_n2[prodi[0]]\
                                /gain_theory_n2[prodj[0]]\
                                /16384

    dd0_all_nogain_n3[:,prodii] = dd0_all_n3[:,prodii] * (2**44)**2 \
                                /gain_theory_n3[prodi[0]]\
                                /gain_theory_n3[prodj[0]]\
                                /16384

    #dd0_all_nogain_n4[:,prodii] = dd0_all_n4[:,prodii] * (2**44)**2 \
    #                            /gain_theory_n4[prodi[0]]\
    #                            /gain_theory_n4[prodj[0]]\
    #                            /16384


In [ ]:
def get_lagspectra_and_closure_residual(dd,prod_inds,Nfft = 8192,fsnew = 3.2e9, nyqflip=False):
    #if not nyqflip:
    #    all_lagspectra = np.fft.ifft(dd[:,prod_inds], n=Nfft*2, axis=0)  # length equals the length of fft    
    #else:

    mask_ = np.ones(8192)
    #mask_[2000:6000] = 1
    all_lagspectra = np.fft.ifft(dd[:,prod_inds] * mask_[:,None], n=Nfft*2, axis=0)  # length equals the length of fft    
    # Center zero lag and build lag axis
    #if nyqflip:
    #    all_lagspectra = np.fft.fftshift(all_lagspectra[::-1,:],axes=0) 
    #else:
    #    
    all_lagspectra = np.fft.fftshift(all_lagspectra,axes=0) 
    
    lags_samp = np.arange(-Nfft, Nfft)      # lag in samples

    # lag window
    lag_idx, _lags_disp = _lag_window_indices(lags_samp, [-170, 170])
    
    peak_lags = np.argmax(np.abs(all_lagspectra)[lag_idx,:],axis=0) 
    max_lagspectra = all_lagspectra[lag_idx,:][peak_lags]

    peak_lags = lags_samp[lag_idx][peak_lags]
    
    tau_ij_ns = np.zeros((64,64))
    maxlagspec_ij_sn = np.zeros((64,64))    
    pairs = [(i, j) for i in range(64) for j in range(i + 1,64)]

    peak_lags_ns = np.zeros(len(pairs))

    lagspec_rms = np.abs(all_lagspectra[lag_idx,:])
    lagspec_rms = np.median(np.abs(lagspec_rms -np.median(lagspec_rms,axis=0)[None,:]),axis=0)
    
    for p0, (i, j) in enumerate(pairs):    
        spec_i  = np.abs(all_lagspectra[lag_idx,p0])
        t_i = np.argmax(spec_i)
        tau_i = lags_samp[lag_idx][t_i] + parabolic_subbin(spec_i, t_i)
        tau_i = (tau_i / fsnew) * 1e9

        
        peak_lags_ns[p0] = tau_i

        #if nyqflip:        
        #    tau_i = -tau_i
        #if tau_i < 0:
        #    tau_i *= -1
        tau_ij_ns[i, j] = tau_i
        tau_ij_ns[j, i] = -tau_i

        maxlagspec_ij_sn[i, j] = np.abs(spec_i)[t_i] / lagspec_rms[p0]
        #maxlagspec_ij_sn[j, i] = np.abs(spec_i)[t_i] / lagspec_rms[p0]        
        
        if j >= 32 and j <48 and i >= 16 and i < 32:
            tau_ij_ns[i,j] *= -1  
            tau_ij_ns[j,i] *= -1  
            
    
    tau0 = tau_ij_ns[0, :]

    # residuals for i<j, i,j != 0
    res_ij_adc_ns = np.zeros((64,64))
    
    residuals = []
    for i in range(1, 64):
        for j in range(i + 1, 64):
            tau_meas = tau_ij_ns[i, j]
            tau_pred = tau0[j] - tau0[i]

            #if np.abs(tau_meas - tau_pred) /1e9 * fsnew > 1:
            #    print(np.abs(tau_meas - tau_pred) /1e9 * fsnew, i, j)
            residuals.append(tau_meas - tau_pred)
            res_ij_adc_ns[i,j] = tau_meas - tau_pred

    
    residuals = np.asarray(residuals, dtype=np.float64)
    expected = (64 - 1) * (64 - 2) // 2

    for p0, (i, j) in enumerate(pairs):    
        tau_ij_ns[j, i] = 0

    plt.figure(figsize=(8, 4),dpi = 100)
    plt.hist(peak_lags_ns, bins=120)
    plt.xlabel("Peak delay [ns]")
    plt.ylabel("Count (products)")
    #plt.title("Peak-delay histogram (unique pairs; P={}, seg={})".format(len(pairs), nseg_eff))
    plt.grid(True)
    #plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,8))    
    plt.imshow( maxlagspec_ij_sn,origin='lower',extent=[0,64,0,64],
               interpolation='None', cmap = 'viridis')#,vmin=min_v,vmax=max_v)#,vmax=300,vmin=-300)
    plt.title('Max Peak FFT [S/N]')
    plt.xlabel('Input')
    plt.ylabel('Input')
    for xii in range(8):
        plt.axvline(x=8*xii,c='k')    
        plt.axhline(y=8*xii,c='k')
    plt.colorbar()
    #plt.tight_layout()
    plt.show()


    
    vmin, vmax = float(np.min(peak_lags)), float(np.max(peak_lags))
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.cm.coolwarm
    
    N_prods = all_lagspectra.shape[1]
    
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    for p in range(N_prods):
        ax.plot(
            _lags_disp ,
            np.abs(all_lagspectra[lag_idx,p]),
            color=cmap(norm(float(peak_lags[p]))),
            alpha=0.15,
            linewidth=1.0,
        )
    ax.set_xlabel("Lag [samples]")
    ax.set_ylabel("|cross-correlation|")
    ax.set_title("Lag spectra (sorted by peak lag; color = peak lag)")
    ax.grid(True)
    plt.show()

    plt.figure(figsize=(8,8))
    plt.imshow( tau_ij_ns,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
    plt.title('Max delay FFT [ns]')
    plt.xlabel('Input')
    plt.ylabel('Input')
    for xii in range(8):
        plt.axvline(x=8*xii,c='k')
        plt.axhline(y=8*xii,c='k')    
    plt.colorbar()
    #plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,8))
    plt.imshow( np.abs(res_ij_adc_ns * fsnew) / 1e9, origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
    plt.title('Closure Residual [ns]')
    plt.xlabel('Input')
    plt.ylabel('Input')
    plt.colorbar()
    for xii in range(8):
        plt.axvline(x=8*xii,c='k')    
        plt.axhline(y=8*xii,c='k')
    #plt.tight_layout()
    plt.show()


    #bins_ = np.linspace(-4.,4.,401)# / fsnew) * 1e9
    bins_ = 120
    # plot histogram
    plt.figure(figsize=(8, 4))
    plt.hist( ( residuals * fsnew) / 1e9, bins=bins_)
    plt.xlabel("Closure Residual [samples]")
    plt.ylabel("Count")
    #plt.title("Closure residuals (ref=0 in selected set; N={}, seg={}, refine={})".format(
    #    M, nseg_eff, refine
    #))
    plt.grid(True)
    #plt.tight_layout()
    plt.show()
    

    res_ij_adc_ns_board = np.zeros((8,64))
     
    residuals_board = np.zeros((8, (7 * 6 )//2))
    for i in range(0,8):
        tau0 = tau_ij_ns[i*8, :]
        ci = 0
        for j in range(1,8):
            for k in range(j + 1, 8):
                tau_meas = tau_ij_ns[i*8 +j, i*8 +k]
                tau_pred = tau0[i*8 +k] - tau0[i*8+j]
    
                residuals_board[i, ci]= tau_meas - tau_pred
                ci += 1
                
                res_ij_adc_ns_board[j, i*8 +k] =  tau_meas - tau_pred

    plt.figure(figsize=(3*7,3))
    plt.imshow( res_ij_adc_ns_board ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
    plt.title('Board Closure Residuals [ns]')
    plt.xlabel('Input')
    plt.ylabel('Input')
    for xii in range(8):
        plt.axvline(x=8*xii,c='k')
    plt.colorbar(fraction=0.2)
    #plt.tight_layout()
    plt.show()    


    return {
        "lags_disp_samples": lags_samp,
        "corr_mag_disp": all_lagspectra,
        "peak_lags_ns": peak_lags_ns,
        "max_peak_ij": maxlagspec_ij_sn,
        "max_tau_ij": tau_ij_ns,        
        "closure_res": residuals,                
        "closure_res_ij": res_ij_adc_ns,                
        "closure_res_board": residuals_board,                
        "closure_res_board_ij": res_ij_adc_ns_board,
        "sel_idx": lag_idx,
        }



In [ ]:
bar1 = get_lagspectra_and_closure_residual(dd0_all_nogain_n1, prod_inds)

In [ ]:
bar2 = get_lagspectra_and_closure_residual(dd0_all_nogain_n2, prod_inds,nyqflip=True)

In [ ]:
bar3 = get_lagspectra_and_closure_residual(dd0_all_nogain_n3, prod_inds)

In [ ]:
bar4 = get_lagspectra_and_closure_residual(dd0_all_nogain_n4, prod_inds,nyqflip=True)

In [ ]:
plt.close('all')

In [ ]:
plt.figure()
plt.hist(bar1['peak_lags_ns'],bins=120,histtype='step',label='New Firm. 1')
plt.hist(bar2['peak_lags_ns'],bins=120,histtype='step',label='New Firm. 2')
plt.hist(bar3['peak_lags_ns'],bins=120,histtype='step',label='New Firm. 3')
#plt.hist(bar4['peak_lags_ns'],bins=120,histtype='step',label='Fourth Nyq. Zone')
plt.xlabel("Peak delay [ns]")
plt.ylabel("Count (products)")
#plt.title("Peak-delay histogram (unique pairs; P={}, seg={})".format(len(pairs), nseg_eff))
plt.grid(True)
plt.legend(loc=0)
#plt.tight_layout()
plt.show()



bins_ = np.linspace(-10,10,201)# / fsnew) * 1e9
plt.figure()
plt.hist((bar1['closure_res']* 3.2e9) / 1e9,bins=100,histtype='step',label='New Firm. 1')
plt.hist((bar2['closure_res']* 3.2e9) / 1e9 ,bins=100,histtype='step',label='New Firm. 2')
plt.hist((bar3['closure_res']* 3.2e9) / 1e9 ,bins=100,histtype='step',label='New Firm. 3')
#plt.hist((bar4['closure_res']* 3.2e9) / 1e9 ,bins=100,histtype='step',label='Fourth Nyq. Zone')
plt.xlabel("Closure residual [samples]")
plt.ylabel("Count")
#plt.yscale('log')
plt.grid(True)
#plt.tight_layout()
plt.legend(loc=0)
plt.show()

print('New Firm. 1 | Closure Residuals [ps]| mean : ', np.mean(bar1['closure_res']*1e3), ', stdev. : ', np.std(bar1['closure_res']*1e3)  )
print('New Firm. 2 | Closure Residuals [ps]| mean : ', np.mean(bar2['closure_res']*1e3), ', stdev. : ', np.std(bar2['closure_res']*1e3)  )
print('New Firm. 3 | Closure Residuals [ps]| mean : ', np.mean(bar3['closure_res']*1e3), ', stdev. : ', np.std(bar3['closure_res']*1e3)  )
#print('4th Nyq Zone | Closure Residuals [ps]| mean : ', np.mean(bar4['closure_res']*1e3), ', stdev. : ', np.std(bar4['closure_res']*1e3)  )

In [ ]:
plt.figure(figsize=(3*7,3))
plt.imshow( (bar1['closure_res_board_ij'])*1e3 ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
plt.title('Board Closure Residual [ps] | New Firm. 1')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()


plt.figure(figsize=(3*7,3))

plt.imshow( (bar2['closure_res_board_ij'])*1e3 ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
plt.title('Board Closure Residual [ps] | New Firm. 2')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()



plt.figure(figsize=(3*7,3))

plt.imshow( (bar3['closure_res_board_ij'])*1e3 ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
plt.title('Board Closure Residual [ps] | New Firm. 3')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()


#plt.figure(figsize=(3*7,3))

#plt.imshow( (bar3['closure_res_board_ij'])*1e3 ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
#plt.title('Board Closure Residual [ps] | Nyq. 3')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()



#plt.figure(figsize=(3*7,3))
#plt.imshow( (bar4['closure_res_board_ij'])*1e3 ,origin='lower',extent=[0,64,0,8],interpolation='None', cmap = 'viridis')
#plt.title('Board Closure Residual [ps] | Nyq. 4')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()




In [ ]:
plt.figure()
plt.plot(np.sort(bar1['max_tau_ij'][2,:]*3.2e9/1e9),label='New Firm. 1 (ref)')
plt.plot(np.sort(bar2['max_tau_ij'][2,:]*3.2e9/1e9),label='New Firm. 2 (power cycle)')
plt.plot(np.sort(bar3['max_tau_ij'][2,:]*3.2e9/1e9),label='New Firm. 3 (+1 day)')
#plt.plot(np.sort(bar3['max_tau_ij'][2,:]*3.2e9/1e9),label='Nyq 3 ')
#plt.plot(np.sort(bar4['max_tau_ij'][2,:]*3.2e9/1e9),label='Nyq 4 ')
#plt.plot(np.sort(bar4['max_tau_ij'][2,:]*3.2e9/1e9),label='4')
plt.ylabel('Time Lag [Samples]')
plt.xlabel('Time lag Sorted Input')
plt.title('Slice input 2')
plt.legend(loc=0)
plt.show()



In [ ]:


plt.figure(figsize=(8,8))
plt.imshow( (bar1['max_tau_ij'] - bar2['max_tau_ij'] ) ,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT Diff [ns] | New Firm. 1 - 2')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()

plt.figure(figsize=(8,8))
plt.imshow( (bar1['max_tau_ij'] - bar3['max_tau_ij'] ) ,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT Diff [ns] | New Firm. 1 - 3')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()


plt.figure(figsize=(8,8))
plt.imshow( (bar3['max_tau_ij'] - bar2['max_tau_ij'] ) ,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT Diff [ns] | New Firm. 3 - 2')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
#plt.tight_layout()
plt.show()

#plt.figure(figsize=(8,8))
#plt.imshow( (bar1['max_tau_ij'] - bar2['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT Diff [ns] | Power Cycle 1')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()


#plt.figure(figsize=(8,8))
#plt.imshow( (bar1['max_tau_ij'] - bar3['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT Diff [ns] | Power Cycle 2')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()


In [ ]:
plt.figure(figsize=(8,8))
plt.imshow( np.abs(bar2['max_tau_ij'] - bar1['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT [ns] |  Nyq 2 - Nyq 1 ')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.tight_layout()
plt.show()

#plt.figure(figsize=(8,8))
#plt.imshow( np.abs(bar3['max_tau_ij'] - bar1['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT [ns] |  Nyq 3 - Nyq 1 ')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#for xii in range(8):
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()


#plt.figure(figsize=(8,8))
#plt.imshow( np.abs(bar4['max_tau_ij'] - bar1['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT [ns] |  Nyq 4 - Nyq 1 ')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#for xii in range(8):
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()

#plt.figure(figsize=(8,8))
#plt.imshow( np.abs(bar3['max_tau_ij'] - bar2['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT [ns] |  Nyq 3 - Nyq 2 ')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#for xii in range(8):
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()

#plt.figure(figsize=(8,8))
#plt.imshow( np.abs(bar4['max_tau_ij'] - bar2['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT [ns] |  Nyq 4 - Nyq 2 ')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#for xii in range(8):
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()

#plt.figure(figsize=(8,8))
#plt.imshow( np.abs(bar4['max_tau_ij'] - bar3['max_tau_ij']),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
#plt.title('Max delay FFT [ns] |  Nyq 4 - Nyq 3 ')
#plt.xlabel('Input')
#plt.ylabel('Input')
#for xii in range(8):
#    plt.axvline(x=8*xii,c='k')
#for xii in range(8):
#    plt.axhline(y=8*xii,c='k')
#plt.colorbar()
#plt.tight_layout()
#plt.show()



In [ ]:
plt.figure(figsize=(8,8))    
plt.imshow( bar2['max_peak_ij'] / bar1['max_peak_ij'] ,origin='lower',extent=[0,64,0,64],
           interpolation='None', cmap = 'viridis')#,vmin=min_v,vmax=max_v)#,vmax=300,vmin=-300)
plt.title('Max Peak FFT [S/N] 2 / 1')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')

for xii in range(8):
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.tight_layout()
plt.show()


plt.figure(figsize=(8,8))    
plt.imshow( bar3['max_peak_ij'] / bar1['max_peak_ij'] ,origin='lower',extent=[0,64,0,64],
           interpolation='None', cmap = 'viridis')#,vmin=min_v,vmax=max_v)#,vmax=300,vmin=-300)
plt.title('Max Peak FFT [S/N] 3 / 1')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')

for xii in range(8):
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,8))    
plt.imshow( bar3['max_peak_ij'] / bar2['max_peak_ij'] ,origin='lower',extent=[0,64,0,64],
           interpolation='None', cmap = 'viridis')#,vmin=min_v,vmax=max_v)#,vmax=300,vmin=-300)
plt.title('Max Peak FFT [S/N] 3 / 2')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')

for xii in range(8):
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.tight_layout()
plt.show()



In [ ]:
#ind_ii = 1004
ind_ii = 167

print(np.where(prod_map_n1 == prod_inds[ind_ii]))
plt.figure()
plt.plot( np.arange(-170,171), np.abs(bar2['corr_mag_disp'][bar2['sel_idx'],ind_ii]),'-o', label='Trial 1 (Reference)' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot( np.arange(-170,171), np.abs(bar1['corr_mag_disp'][bar1['sel_idx'],ind_ii]), '-o', label='Trial 2 (Power Cycle)' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot( np.arange(-170,171), np.abs(bar3['corr_mag_disp'][bar3['sel_idx'],ind_ii]),'-o', label='Trial 3 (Resync)' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.arange(-170,171), np.abs(bar4['corr_mag_disp'][bar4['sel_idx'],ind_ii]), label='4' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( foo['spectra']['corr_mag_disp'][ind_ii] / lagspec_adc_rms[ind_ii],label='Raw ADC')
plt.title(f'Input {np.where(prod_map_n1 == prod_inds[ind_ii])[0][0]} X {np.where(prod_map_n1 == prod_inds[ind_ii])[1][0]}')
plt.legend(loc=0)
#plt.yscale('log')
plt.xlabel('Time Lag [samples]')
plt.ylabel('Correlation [S/N]')
plt.show()


ind_ii = 1657

print(np.where(prod_map_n1 == prod_inds[ind_ii]))
plt.figure()
plt.plot( np.arange(-170,171), np.abs(bar1['corr_mag_disp'][bar1['sel_idx'],ind_ii]), '-o', label='Trial 1' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot( np.arange(-170,171), np.abs(bar2['corr_mag_disp'][bar2['sel_idx'],ind_ii]),'-o', label='Trial 2' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot( np.arange(-170,171), np.abs(bar3['corr_mag_disp'][bar3['sel_idx'],ind_ii]),'-o', label='Trial 3' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.arange(-170,171), np.abs(bar4['corr_mag_disp'][bar4['sel_idx'],ind_ii]), label='4' )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( foo['spectra']['corr_mag_disp'][ind_ii] / lagspec_adc_rms[ind_ii],label='Raw ADC')
plt.title(f'Input {np.where(prod_map_n1 == prod_inds[ind_ii])[0][0]} X {np.where(prod_map_n1 == prod_inds[ind_ii])[1][0]}')
plt.legend(loc=0)
#plt.yscale('log')
plt.xlabel('Time Lag [samples]')
plt.ylabel('Correlation [S/N]')
plt.show()



In [ ]:
ind_ii = 1004
print(dd0_all_n1.shape,prod_inds[ind_ii])
print(np.where(prod_map_n1 == prod_inds[ind_ii]))
plt.figure()
plt.plot(np.abs(dd0_all_nogain_n1[:,prod_inds[ind_ii]]) ,label='nyq 1'  )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot(np.abs(dd0_all_nogain_n2[:,prod_inds[ind_ii]]) ,label='nyq 2'  )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot(np.abs(dd0_all_nogain_n3[:,prod_inds[ind_ii]]) ,label='nyq 3'  )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot(np.abs(dd0_all_nogain_n4[:,prod_inds[ind_ii]]) ,label='nyq 4'  )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.arange(-170,171), np.abs(bar2['corr_mag_disp'][bar2['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.arange(-170,171), np.abs(bar3['corr_mag_disp'][bar3['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.arange(-170,171), np.abs(bar4['corr_mag_disp'][bar4['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( foo['spectra']['corr_mag_disp'][ind_ii] / lagspec_adc_rms[ind_ii],label='Raw ADC')
plt.title(f'Input {np.where(prod_map_n1 == prod_inds[ind_ii])[0][0]} X {np.where(prod_map_n1 == prod_inds[ind_ii])[1][0]}')
plt.legend(loc=0)
plt.yscale('log')
plt.xlabel('Freq [samples]')
plt.ylabel('Correlation [S/N]')
plt.show()


## ADC Products

In [ ]:
import matplotlib as mpl
mpl.rcParams["savefig.dpi"] = 100
mpl.rcParams["figure.dpi"]  = 100

foo1 = lag_spectra_and_closure_report(
    d_adc_n1, streams_n1,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=1,
#     Nseg=None,
#     Nfft=None,
    refine=True,
    up_factor=1,
    lag_window_spectra=(-170, 170),
    lag_window_closure=(-170, 170),
    hist_bins_spectra=60,
    hist_bins_closure=int(2*np.sqrt(2016)),
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.2,
    heatmap_logscale=False,
    show_debug=True)




In [ ]:
foo2 = lag_spectra_and_closure_report(
    d_adc_n2, streams_n2,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=1,
#     Nseg=None,
#     Nfft=None,
    refine=True,
    up_factor=1,
    lag_window_spectra=(-170, 170),
    lag_window_closure=(-170, 170),
    hist_bins_spectra=60,
    hist_bins_closure=int(2*np.sqrt(2016)),
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.2,
    heatmap_logscale=False,
    show_debug=True)


In [ ]:
foo3 = lag_spectra_and_closure_report(
    d_adc_n3, streams_n3,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=1,
#     Nseg=None,
#     Nfft=None,
    refine=True,
    up_factor=1,
    lag_window_spectra=(-170, 170),
    lag_window_closure=(-170, 170),
    hist_bins_spectra=60,
    hist_bins_closure=int(2*np.sqrt(2016)),
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.2,
    heatmap_logscale=False,
    show_debug=True)


In [ ]:
foo4 = lag_spectra_and_closure_report(
    d_adc_n4, streams_n4,
    boards=None, chans=None,
    fs=3.2e9, frame_idx=0,
    n_segments=1,
#     Nseg=None,
#     Nfft=None,
    refine=True,
    up_factor=1,
    lag_window_spectra=(-170, 170),
    lag_window_closure=(-170, 170),
    hist_bins_spectra=60,
    hist_bins_closure=int(2*np.sqrt(2016)),
    show_spectra=True,
    show_heatmap=True,
    spectra_max=None,
    spectra_alpha=0.2,
    heatmap_logscale=False,
    show_debug=True)


In [ ]:
ind_ii = 160 #1003
plt.figure()
plt.plot( np.abs(foo1['spectra']['corr_mag_disp'][ind_ii]) ,label='1')#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
plt.plot( np.abs(foo2['spectra']['corr_mag_disp'][ind_ii]) ,label='2')#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.abs(foo3['spectra']['corr_mag_disp'][ind_ii]) ,label='3')#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.abs(foo4['spectra']['corr_mag_disp'][ind_ii]) ,label='4')#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')

#plt.plot( np.abs(bar1['corr_mag_disp'][bar1['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.abs(bar2['corr_mag_disp'][bar2['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( np.abs(bar3['corr_mag_disp'][bar3['sel_idx'],ind_ii]) )#/ lagspec_uncal_rms[ind_ii],label='Calib. FFT')
#plt.plot( foo['spectra']['corr_mag_disp'][ind_ii] / lagspec_adc_rms[ind_ii],label='Raw ADC')
plt.title(f'Input {np.where(prod_map_n1 == prod_inds[ind_ii])[0][0]} X {np.where(prod_map_n1 == prod_inds[ind_ii])[1][0]}')
plt.legend(loc=0)
plt.yscale('log')
plt.xlabel('Time Lag [samples]')
plt.ylabel('Correlation [S/N]')
plt.show()


In [ ]:
tau_ij_adc_ns_n1 = np.zeros((64,64))
tau_ij_adc_ns_n2 = np.zeros((64,64))
#tau_ij_adc_ns_n3 = np.zeros((64,64))
#tau_ij_adc_ns_n4 = np.zeros((64,64))

maxpeak_adc_ij_n1 = np.zeros((64, 64))
maxpeak_adc_ij_n2 = np.zeros((64, 64))
#maxpeak_adc_ij_n3 = np.zeros((64, 64))
#maxpeak_adc_ij_n4 = np.zeros((64, 64))


pairs = [(i, j) for i in range(64) for j in range(i + 1,64)]
for p0, (i, j) in enumerate(pairs):
    tau_ij_adc_ns_n1[i,  j] =  foo1['spectra']['peak_tau_ns'][p0] 

    tau_ij_adc_ns_n2[i,  j] =  foo2['spectra']['peak_tau_ns'][p0]

#    tau_ij_adc_ns_n3[i,  j] =  foo3['spectra']['peak_tau_ns'][p0]

#    tau_ij_adc_ns_n4[i,  j] =  foo4['spectra']['peak_tau_ns'][p0]

    spec_i = foo1['spectra']['corr_mag_disp'][p0]
    sn =  np.amax(spec_i) 
    sn /= np.median(np.abs(spec_i- np.median(spec_i))) 
    maxpeak_adc_ij_n1[i,  j] = sn

    spec_i = foo2['spectra']['corr_mag_disp'][p0]
    sn =  np.amax(spec_i) 
    sn /= np.median(np.abs(spec_i- np.median(spec_i))) 
    maxpeak_adc_ij_n2[i,  j] = sn

#    spec_i = foo3['spectra']['corr_mag_disp'][p0]
#    sn =  np.amax(spec_i) 
#    sn /= np.median(np.abs(spec_i- np.median(spec_i))) 
#    maxpeak_adc_ij_n3[i,  j] = sn

#    spec_i = foo4['spectra']['corr_mag_disp'][p0]
#    sn =  np.amax(spec_i) 
#    sn /= np.median(np.abs(spec_i- np.median(spec_i))) 
#    maxpeak_adc_ij_n4[i,  j] = sn





In [ ]:
prod_map_n1[18,42]
print(np.where(np.array(prod_inds) == 1023))

## Comparisons

### Max peak

In [ ]:
pairs_adc = _pair_list(64) 
pairs_zeros_adc = [(i,i) for i in range(64)]#_pair_list(64) 


In [ ]:

plt.figure(figsize=(8,8))
plt.imshow( (bar1['max_peak_ij'] / maxpeak_adc_ij_n1 ),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')#,vmax=300,vmin=-300)
plt.title('Max Peak FFT/ADC Nyq 1')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.show()

plt.figure(figsize=(8,8))
plt.imshow( (bar2['max_peak_ij'] / maxpeak_adc_ij_n2 ),origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')#,vmax=300,vmin=-300)
plt.title('Max Peak FFT/ADC Nyq 2')
plt.xlabel('Input')
plt.ylabel('Input')
for xii in range(8):
    plt.axvline(x=8*xii,c='k')
    plt.axhline(y=8*xii,c='k')
plt.colorbar()
plt.show()




In [ ]:

bins_ = np.linspace(0,3,150)
plt.figure()
plt.hist( (bar1['max_peak_ij'] / maxpeak_adc_ij_n1 ).ravel(),bins=bins_,label='New Firm. 1',histtype='step')
plt.hist( (bar2['max_peak_ij'] / maxpeak_adc_ij_n2 ).ravel(),bins=bins_,label='New Firm. 2',histtype='step')
#plt.hist( (bar3['max_peak_ij'] / maxpeak_adc_ij_n3 ).ravel(),bins=bins_,label='Nyq 3',histtype='step')
#plt.hist( (bar4['max_peak_ij'] / maxpeak_adc_ij_n4 ).ravel(),bins=bins_,label='Nyq 4',histtype='step')
plt.xlabel('Max Peak FFT/ADC S/N')
plt.ylabel('Counts')
plt.legend(loc=0)
plt.tight_layout()
plt.show()



### Max delay

In [ ]:
plt.figure(figsize=(8,8))
plt.imshow( np.abs(tau_ij_adc_ns_n1 - bar1['max_tau_ij'] ) *1e3 ,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT - ADC | New Firm. 1 | [ps]')
plt.xlabel('Input')
plt.ylabel('Input')
plt.colorbar()
plt.show()

plt.figure(figsize=(8,8))
plt.imshow( np.abs(tau_ij_adc_ns_n2 - bar2['max_tau_ij'])*1e3 ,origin='lower',extent=[0,64,0,64],interpolation='None', cmap = 'viridis')
plt.title('Max delay FFT - ADC | New Firm. 2 | [ps]')
plt.xlabel('Input')
plt.ylabel('Input')
plt.colorbar()
plt.show()



### Closure Residual

In [ ]:

bins_ = np.linspace(-0.5,0.5,101)# / fsnew) * 1e9
# plot histogram
plt.figure(figsize=(8, 4))
plt.hist( ( residuals_uncal * fsnew) / 1e9, bins=bins_,alpha=0.5,label='Calib. FFT')
plt.hist( ( residuals_adc * fsnew) / 1e9, bins=bins_,alpha=0.5,label='ADC')
plt.xlabel("Closure residual [samples]")
plt.ylabel("Count")
plt.legend(loc=0)
#plt.title("Closure residuals (ref=0 in selected set; N={}, seg={}, refine={})".format(
#    M, nseg_eff, refine
#))
plt.grid(True)
plt.tight_layout()
plt.show()

#residuals_adc

# Beamformer

## Correlator Products

In [ ]:
def get_beamformed_prods(dd, tau_ns, prod_map, n_nyqzone=1, aperture_m=100):
    C0 = 299_792_458.0  # m/s    
    pairs = _pair_list(64)    
    M = 2**14 # samples
    fs = 3200 *1e6 # Hz
    f_res = fs / M

    if n_nyqzone % 2 == 0:
        fnyq = fs * n_nyqzone / 2
    else:
        fnyq = fs * (n_nyqzone - 1) / 2

    fnyq_sign = ( ( n_nyqzone % 2 ) * 2 - 1 )
    
    freq = fnyq + fnyq_sign * f_res * np.arange(M//2)    

    peak_lags_ns_x = []
    peak_lags_ns_y = []
    
    x_inds = []
    y_inds = []
    for p0,(pi,pj) in enumerate(pairs):    
        if p0 < 32:
            peak_lags_ns_x.append(tau_ns[p0])
            x_inds.append(prod_map[pi,pj])
        elif p0 > 32 and p0 < 64:
            peak_lags_ns_y.append(tau_ns[p0])            
            y_inds.append(prod_map[pi,pj])

    corr_prods_x = dd[:,x_inds]
    corr_prods_y = dd[:,y_inds]
    
    peak_lags_ns_x = np.array(peak_lags_ns_x)
    peak_lags_ns_y = np.array(peak_lags_ns_y)

    x_m = build_effective_geometry_from_tau(peak_lags_ns_x , aperture_m=aperture_m, center=True)
    y_m = build_effective_geometry_from_tau(peak_lags_ns_y , aperture_m=aperture_m, center=True)


    idx_x = np.argsort(x_m )
    idx_y = np.argsort(y_m )


    x2 = x_m[idx_x]
    y2 = y_m[idx_y]

    # ----- plot L geometry -----
    Xcoords = np.concatenate([x2, np.zeros_like(y2)])
    Ycoords = np.concatenate([np.zeros_like(x2), y2])
    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
    ax.scatter(Xcoords, Ycoords, s=18)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("East-West x [m]")
    ax.set_ylabel("North-South y [m]")
    ax.set_title("L-array geometry (scaled)")
    ax.grid(True)
    plt.show()


    theta_deg = np.linspace(-90, 90, 721)
    

    #x_m = 100
    thetas = np.deg2rad(np.asarray(theta_deg, dtype=float))
    Paz = np.zeros_like(thetas, dtype=np.float64)
    Pel = np.zeros_like(thetas, dtype=np.float64)
    
    #iterator = list(enumerate(thetas))
    #iterator = tqdm(iterator, total=len(thetas), unit="theta", leave=True)

    
    for ti, th in enumerate(thetas):
        tau = (x_m * np.sin(th)) / C0
        W = np.exp(fnyq_sign*1j * 2 * np.pi * freq[None, :] * tau[:, None]).astype(np.complex64)
    
        acc = 0.0
        Yf = np.sum(W * corr_prods_x.T, axis=0)
        acc += np.sum(np.abs(Yf) ** 2)
        Paz[ti] = acc 
    
        tau = (y_m * np.sin(th)) / C0
        W = np.exp(fnyq_sign*1j * 2 * np.pi * freq[None, :] * tau[:, None]).astype(np.complex64)
    
        acc = 0.0
        Yf = np.sum(W * corr_prods_y.T, axis=0)
        acc += np.sum(np.abs(Yf) ** 2)
        Pel[ti] = acc 
    
    
    tx = np.asarray(theta_deg, dtype=float)
    ty = np.asarray(theta_deg, dtype=float)


    u_scan = np.sin(np.deg2rad(tx))
    v_scan = np.sin(np.deg2rad(ty))
    
    # separable peaks
    iu = int(np.argmax(Paz))
    iv = int(np.argmax(Pel))
    peak_tx = float(tx[iu])
    peak_ty = float(ty[iv])
    peak_u = float(u_scan[iu])
    peak_v = float(v_scan[iv])
    peak_P_uv = float(Paz[iu] * Pel[iv])
    
    # ----- EW/NS heatmap in θx vs θy -----
    P_uv = Paz[None, :] * Pel[:, None]  # (len(ty), len(tx))
    P_uv_max = float(np.max(P_uv))
    #if heatmap_db:
    Z_uv = 10.0 * np.log10(P_uv / P_uv_max)
    peak_uv_db = 10.0 * np.log10(peak_P_uv / P_uv_max)
    cbar_uv = "Power [dB, norm]"
    #else:
    #Z_uv = P_uv / P_uv_max
    #peak_uv_db = 10.0 * np.log10(peak_P_uv / P_uv_max)
    #cbar_uv = "Power [linear, norm]"
    
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    im = ax.imshow(
        Z_uv, origin="lower", aspect="auto",
        extent=[tx[0], tx[-1], ty[0], ty[-1]],
        interpolation=None,
    )
    ax.scatter([peak_tx], [peak_ty], s=60, color="red", marker="x")
    ax.set_xlabel("EW scan angle θx°")
    ax.set_ylabel("NS scan angle θy°")
    ax.set_title("EW/NS localization")
    fig.colorbar(im, ax=ax, label=cbar_uv)
    ax.text(
        0.02, 0.98,
        ("Peak:\n"
         "  θx = {:.3f}° , θy = {:.3f}°").format(
            peak_tx, peak_ty),
        transform=ax.transAxes, va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
        fontsize=10,
    )
    plt.show()

    # EW/NS slices through peak
    fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
    axs[0].plot(tx, Z_uv[iv, :])
    axs[0].axvline(peak_tx, linestyle="--")
    axs[0].set_title("EW/NS slice: along θx at peak θy")
    axs[0].set_xlabel("θx°")
    axs[0].set_ylabel(cbar_uv)
    axs[0].grid(True)
    
    axs[1].plot(ty, Z_uv[:, iu])
    axs[1].axvline(peak_ty, linestyle="--")
    axs[1].set_title("EW/NS slice: along θy at peak θx")
    axs[1].set_xlabel("θy°")
    axs[1].set_ylabel(cbar_uv)
    axs[1].grid(True)
    plt.show()
    
    # ----- Spherical AZ/EL heatmap via mapping u,v -----
    az_sph = np.asarray(np.linspace(0, 360, 721), dtype=float)
    el_sph = np.asarray(np.linspace(0, 90, 361), dtype=float)
    AZ, EL = np.meshgrid(az_sph, el_sph, indexing="xy")  # (Nel, Naz)
    
    azr = np.deg2rad(AZ)
    elr = np.deg2rad(EL)
    u = np.cos(elr) * np.cos(azr)
    v = np.cos(elr) * np.sin(azr)
    
    def interp1(x, xp, fp):
        return np.interp(x, xp, fp, left=0.0, right=0.0)
    
    Paz_uv = interp1(u.ravel(), u_scan, Paz).reshape(u.shape)
    Pel_uv = interp1(v.ravel(), v_scan, Pel).reshape(v.shape)
    P_sph = Paz_uv * Pel_uv
    
    P_sph_max = float(np.max(P_sph))
    peak_flat = int(np.argmax(P_sph))
    peak_el_i, peak_az_i = np.unravel_index(peak_flat, P_sph.shape)
    peak_az = float(az_sph[peak_az_i])
    peak_el = float(el_sph[peak_el_i])
    peak_P_sph = float(P_sph[peak_el_i, peak_az_i])
    
    Z_sph = 10.0 * np.log10(P_sph / P_sph_max)
    peak_sph_db = 10.0 * np.log10(peak_P_sph / P_sph_max)
    cbar_sph = "Power [dB, norm]"
    #else:
    #    Z_sph = P_sph / P_sph_max
    #    peak_sph_db = 10.0 * np.log10(peak_P_sph / P_sph_max)
    #    cbar_sph = "Power [linear, norm]"
    
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    im = ax.imshow(
        Z_sph, origin="lower", aspect="auto",
        extent=[az_sph[0], az_sph[-1], el_sph[0], el_sph[-1]],
        interpolation=None,
    )
    ax.scatter([peak_az], [peak_el], s=60, color="red", marker="x")
    ax.set_xlabel("Azimuth [deg]")
    ax.set_ylabel("Elevation [deg]")
    ax.set_title("Spherical AZ/EL localization (mapped from EW/NS direction cosines)")
    fig.colorbar(im, ax=ax, label=cbar_sph)
    ax.text(
        0.02, 0.1,
        ("Peak:\n"
         "  az = {:.2f}° , el = {:.2f}°").format(peak_az, peak_el),
        transform=ax.transAxes, va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
        fontsize=10,
    )
    plt.show()
    
    fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
    axs[0].plot(az_sph, Z_sph[peak_el_i, :])
    axs[0].axvline(peak_az, linestyle="--")
    axs[0].set_title("AZ/EL slice: along az at peak el")
    axs[0].set_xlabel("Az [deg]")
    axs[0].set_ylabel(cbar_sph)
    axs[0].grid(True)
    
    axs[1].plot(el_sph, Z_sph[:, peak_az_i])
    axs[1].axvline(peak_el, linestyle="--")
    axs[1].set_title("AZ/EL slice: along el at peak az")
    axs[1].set_xlabel("El [deg]")
    axs[1].set_ylabel(cbar_sph)
    axs[1].grid(True)
    plt.show()

    return {
        "x_sorted_m": x2,
        "y_sorted_m": y2,
        "theta_x_deg": tx,
        "theta_y_deg": ty,
        "Paz": Paz,
        "Pel": Pel,
        "u_scan": u_scan,
        "v_scan": v_scan,
        "P_uv": P_uv,
        "P_sph": P_sph,
        "peak_theta_x_deg": peak_tx,
        "peak_theta_y_deg": peak_ty,
        "peak_u": peak_u,
        "peak_v": peak_v,
        "peak_P_uv": peak_P_uv,
        "peak_az_deg": peak_az,
        "peak_el_deg": peak_el,
        "peak_P_sph": peak_P_sph,
    }



In [ ]:
cres1 = get_beamformed_prods(dd0_all_nogain_n1,bar1['peak_lags_ns'],prod_map_n1)


In [ ]:
cres2 = get_beamformed_prods(dd0_all_nogain_n2,bar2['peak_lags_ns'],prod_map_n1,n_nyqzone=1,aperture_m=100)


In [ ]:
cres3 = get_beamformed_prods(dd0_all_nogain_n3,bar3['peak_lags_ns'],prod_map_n1,n_nyqzone=3)


In [ ]:
cres4 = get_beamformed_prods(dd0_all_nogain_n4,bar4['peak_lags_ns'],prod_map_n1,n_nyqzone=4)


## ADC products

In [ ]:
d_adc_n1.shape

In [ ]:
res1 = azel_and_ewns_heatmaps(
    d_x=d_adc_n1, streams_x=streams_n1,
    #d_y=d2, streams_y=streams,   # each shape (64, ncap, nsamp)    
    fs=3.2e9,
    n_segments=None,
    Nfft=None,
    lag_window=(-200, 200),
    refine=True, up_factor=1,
    aperture_x_m=100.0,
    aperture_y_m=100.0,
    n_nyqzone=1,
    fmin=0, fmax=1.6e9,    
    sort_by_pos=True,
    heatmap_db=True,
    show_progress=True,
)

# res = azel_and_ewns_heatmaps(
#     d=d2, streams=streams,
#     fs=3.2e9,
#     n_segments=None,
#     Nfft=None,
#     lag_window=(-75, 75),
#     refine=True, up_factor=16,
#     aperture_x_m=50.0,
#     aperture_y_m=50.0,
#     fmin=100e6, fmax=1.6e9,
#     sort_by_pos=True,
#     heatmap_db=True,
#     show_progress=True,
# )

In [ ]:
res2 = azel_and_ewns_heatmaps(
    d_x=d_adc_n2[:,:,:], streams_x=streams_n2,
    #d_y=d2, streams_y=streams,   # each shape (64, ncap, nsamp)    
    fs=3.2e9,
    n_segments=None,
    Nfft=None,
    lag_window=(-200, 200),
    refine=True, up_factor=1,
    aperture_x_m=100.0,
    aperture_y_m=100.0,
    n_nyqzone=1,
    fmin=0, fmax=1.6e9,   
#    n_nyqzone=2,
#    fmin=1.6e9, fmax=3.2e9,    
    sort_by_pos=True,
    heatmap_db=True,
    show_progress=True,
)

In [ ]:
res3 = azel_and_ewns_heatmaps(
    d_x=d_adc_n3, streams_x=streams_n3,
    #d_y=d2, streams_y=streams,   # each shape (64, ncap, nsamp)    
    fs=3.2e9,
    n_segments=None,
    Nfft=None,
    lag_window=(-200, 200),
    refine=True, up_factor=1,
    aperture_x_m=100.0,
    aperture_y_m=100.0,
    n_nyqzone=3,
    fmin=3.2e9, fmax=4.8e9,    
    sort_by_pos=True,
    heatmap_db=True,
    show_progress=True,
)

In [ ]:
res4 = azel_and_ewns_heatmaps(
    d_x=d_adc_n4, streams_x=streams_n4,
    #d_y=d2, streams_y=streams,   # each shape (64, ncap, nsamp)    
    fs=3.2e9,
    n_segments=None,
    Nfft=None,
    lag_window=(-200, 200),
    refine=True, up_factor=1,
    aperture_x_m=100.0,
    aperture_y_m=100.0,
    n_nyqzone=4,
    fmin=4.8e9, fmax=6.4e9,    
    #n_nyqzone=1,
    #fmin=10e9, fmax=1.6e9,        
    sort_by_pos=True,
    heatmap_db=True,
    show_progress=True,
)

## Compare

In [ ]:
# EW/NS slices through peak
fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
fig.suptitle('Nyq. 1')
axs[0].plot(cres1['theta_x_deg'], cres1['Paz']/np.nanmedian(cres1['Paz']) ,label='FFT')
axs[0].plot(res1['theta_x_deg'], res1['Paz']/np.nanmedian(res1['Paz']) ,label='ADC')
#axs[0].axvline(peak_tx, linestyle="--")
axs[0].set_title("EW/NS slice: along θx at peak θy")
axs[0].set_xlabel("θx°")
axs[0].set_ylabel('Corr S/N')
axs[0].legend(loc=0)
axs[0].grid(True)

axs[1].plot(cres1['theta_y_deg'], cres1['Pel']/np.nanmedian(cres1['Pel']) ,label='FFT')
axs[1].plot(res1['theta_y_deg'], res1['Pel']/np.nanmedian(res1['Pel']) ,label='ADC')
#axs[1].axvline(peak_ty, linestyle="--")
axs[1].set_title("EW/NS slice: along θy at peak θx")
axs[1].set_xlabel("θy°")
axs[1].legend(loc=0)
axs[1].set_ylabel('Corr S/N')
axs[1].grid(True)
plt.show()

# EW/NS slices through peak
fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
fig.suptitle('Nyq. 2')
axs[0].plot(cres2['theta_x_deg'], cres2['Paz']/np.nanmedian(cres2['Paz']) ,label='FFT')
axs[0].plot(res2['theta_x_deg'], res2['Paz']/np.nanmedian(res2['Paz']) ,label='ADC')
#axs[0].axvline(peak_tx, linestyle="--")
axs[0].set_title("EW/NS slice: along θx at peak θy")
axs[0].set_xlabel("θx°")
axs[0].set_ylabel('Corr S/N')
axs[0].legend(loc=0)
axs[0].grid(True)

axs[1].plot(cres2['theta_y_deg'], cres2['Pel']/np.nanmedian(cres2['Pel']) ,label='FFT')
axs[1].plot(res2['theta_y_deg'], res2['Pel']/np.nanmedian(res2['Pel']) ,label='ADC')
#axs[1].axvline(peak_ty, linestyle="--")
axs[1].set_title("EW/NS slice: along θy at peak θx")
axs[1].set_xlabel("θy°")
axs[1].legend(loc=0)
axs[1].set_ylabel('Corr S/N')
axs[1].grid(True)
plt.show()



